<a href="https://colab.research.google.com/github/yagmurdamlakara/enerji-tuketimi-tahmini/blob/main/makale.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 1. KÜTÜPHANELER
# ==========================================

import os
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

warnings.filterwarnings("ignore")

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

NumPy: 2.1.3
Pandas: 2.2.3


In [ ]:
numeric_cols = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",
]

timestamps = pd.to_datetime(
    check_raw["Date"].astype(str).str.strip()
    + " "
    + check_raw["Time"].astype(str).str.strip(),
    format="%d/%m/%Y %H:%M:%S",
    errors="raise"
)

df = check_raw[numeric_cols].apply(pd.to_numeric, errors="coerce")
df.index = pd.DatetimeIndex(timestamps, name="datetime")
df = df.sort_index()

assert len(df) == 2075259, "Tam veri tablosu oluşmadı."
assert not df.index.hasnans
assert not df.index.has_duplicates

print("df satır sayısı:", len(df))
print("Başlangıç:", df.index.min())
print("Bitiş:", df.index.max())

NameError: name 'check_raw' is not defined

In [ ]:
# ==========================================
# 2. RANDOM SEED
# ==========================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Random seed:", SEED)

In [ ]:
DATA_PATH = "/content/household_power_consumption.txt"

print("Dosya mevcut mu?:", os.path.exists(DATA_PATH))

In [ ]:
# ==========================================
# 3. VERİ SETİNİ OKUMA
# ==========================================

df = pd.read_csv(
    DATA_PATH,
    sep=";",
    na_values="?",
    low_memory=False
)

print("Veri boyutu:", df.shape)
df.head()

In [ ]:
print(df.columns.tolist())

In [ ]:
df.head()
df.tail()
df.info()
df.describe()

In [ ]:
print(f"Satır sayısı : {df.shape[0]:,}")
print(f"Sütun sayısı : {df.shape[1]}")

In [ ]:
print("\nEksik değer sayıları:")
print(df.isnull().sum())

In [ ]:
missing_percentage = df.isnull().mean() * 100

print("\nEksik değer yüzdeleri:")
print(missing_percentage)

In [ ]:
# ==========================================
# 4. DATETIME OLUŞTURMA
# ==========================================

df["datetime"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce"
)

In [ ]:
df = df.set_index("datetime")

In [ ]:
df.head()

In [ ]:
df = df.drop(columns=["Date", "Time"])

In [ ]:
df.head()

In [ ]:
df = df.sort_index()
print("Başlangıç:", df.index.min())
print("Bitiş     :", df.index.max())

In [ ]:
duplicate_count = df.index.duplicated().sum()

print("Tekrarlanan zaman damgası sayısı:", duplicate_count)

In [ ]:
numeric_columns = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
df.info()

In [ ]:
missing_count = df.isnull().sum()

missing_percent = (
    df.isnull().sum() /
    len(df) * 100
)

missing_table = pd.DataFrame({
    "Eksik Sayısı": missing_count,
    "Eksik Yüzdesi (%)": missing_percent
})

missing_table

In [ ]:
df[df.isnull().any(axis=1)].head(20)

In [ ]:
rows_with_missing = df.isnull().any(axis=1).sum()

print("Eksik değer içeren satır sayısı:", rows_with_missing)
print(
    "Oran:",
    rows_with_missing / len(df) * 100,
    "%"
)

In [ ]:
time_diffs = df.index.to_series().diff().value_counts().head(10)

print(time_diffs)

In [ ]:
df.resample("h").mean()

In [ ]:
import pandas as pd

DATA_PATH = "/content/household_power_consumption.txt"

# Ham dosyayı yeniden oku.
raw = pd.read_csv(
    DATA_PATH,
    sep=";",
    na_values=["?"],
    low_memory=False
)

# Tarih ve saati açık formatla dönüştür.
datetime_values = pd.to_datetime(
    raw["Date"].astype(str).str.strip()
    + " "
    + raw["Time"].astype(str).str.strip(),
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce"
)

invalid = datetime_values.isna()
print("Geçersiz zaman damgası:", int(invalid.sum()))

# Sorun varsa ilgili satırları göster; sessizce silme.
if invalid.any():
    print(raw.loc[invalid, ["Date", "Time"]].head(20))
    raise ValueError(
        "Tarih/saat dönüşümü başarısız. Yukarıdaki satırları kontrol et."
    )

numeric_cols = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",
]

df = raw[numeric_cols].apply(pd.to_numeric, errors="coerce")
df.index = pd.DatetimeIndex(datetime_values, name="datetime")
df = df.sort_index()

print("Dakikalık veri boyutu:", df.shape)
print("Tekrarlanan zaman damgası:", int(df.index.duplicated().sum()))
print("Başlangıç:", df.index.min())
print("Bitiş:", df.index.max())

In [ ]:
# ==========================================
# 5. SAATLİK VERİ HAZIRLAMA VE KARŞILAŞTIRMA
# ==========================================

MIN_VALID_MINUTES = 60

mean_cols = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
]
sum_cols = [
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",
]
cols = mean_cols + sum_cols

# df: eksikleri doldurulmamış, datetime indeksli dakika verisi.
assert not df.index.hasnans, "Geçersiz zaman damgası var."
assert not df.index.has_duplicates, "Tekrarlanan zaman damgası var."

# Eski yöntem: yalnızca karşılaştırma için.
old_before_fill = df[cols].resample("h").mean()
old_hourly = old_before_fill.interpolate(
    method="time", limit_direction="both"
)

# Geçerli dakika sayıları, doldurma yapılmadan hesaplanır.
valid_minutes = df[cols].resample("h").count()
keep = valid_minutes.ge(MIN_VALID_MINUTES).all(axis=1)

# Yeni yöntem: güç/gerilim/akım ortalama, alt sayaçlar toplam.
new_all_hours = df[mean_cols].resample("h").mean()
new_all_hours[sum_cols] = (
    df[sum_cols].resample("h").sum(min_count=1)
)

# Her sütunda en az 50 gerçek ölçüm bulunan saatler tutulur.
hourly = new_all_hours.loc[keep].copy()

print("Eski yöntemde saat:", len(old_hourly))
print("Yeni yöntemde saat:", len(hourly))
print("Elenen saat:", int((~keep).sum()))
print("Yeni veride eksik hücre:", int(hourly.isna().sum().sum()))
print("\nSütun bazında kalan eksikler:")
print(hourly.isna().sum())
#hourly = df.resample("h").mean()

In [ ]:
import numpy as np
import pandas as pd

LOOKBACK = 24

assert isinstance(hourly.index, pd.DatetimeIndex)
assert not hourly.index.hasnans
assert hourly.index.is_monotonic_increasing
assert not hourly.index.has_duplicates
assert len(hourly) > LOOKBACK

# Her aday: 24 giriş satırı + sonraki 1 hedef satırı.
candidate_count = len(hourly) - LOOKBACK

candidate_positions = (
    np.arange(candidate_count)[:, None]
    + np.arange(LOOKBACK + 1)[None, :]
)

# Adayların tüm zaman damgalarını kontrol et.
times = hourly.index.to_numpy()
window_times = times[candidate_positions]

continuous = (
    np.diff(window_times, axis=1) == np.timedelta64(1, "h")
).all(axis=1)

# Yalnızca kesintisiz örneklerin satır konumlarını sakla.
valid_positions = candidate_positions[continuous]
input_positions = valid_positions[:, :LOOKBACK]
target_positions = valid_positions[:, LOOKBACK]

print("Kalan toplam saat:", len(hourly))
print("Süreklilik kontrolü öncesi aday örnek:", candidate_count)
print("Geçerli 24 saat → 1 saat örneği:", len(valid_positions))
print("Süreklilik nedeniyle elenen örnek:", int((~continuous).sum()))

In [ ]:
#print("Dakikalık veri:", df.shape)
#print("Saatlik veri   :", hourly.shape)

#hourly.head()

In [ ]:
#hourly.isnull().sum()

In [ ]:
#hourly[hourly.isnull().any(axis=1)].head(20)

In [ ]:
#hourly = hourly.interpolate(
#    method="time",
#   limit_direction="both"
#)

In [ ]:
print(hourly.isnull().sum())

In [ ]:
hourly[numeric_columns].describe().T

In [ ]:
for col in numeric_columns:
    print(
        col,
        "Min:", hourly[col].min(),
        "Max:", hourly[col].max()
    )

In [ ]:
(hourly[numeric_columns] < 0).sum()

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(hourly.index, hourly["Global_active_power"])

plt.xlabel("Zaman")
plt.ylabel("Global Active Power (kW)")
plt.title("Saatlik Ortalama Aktif Güç")
plt.grid(alpha=0.3)

plt.show()

In [ ]:
# ==========================================
# MAKALE GRAFİĞİ:
# TÜKETİMİN ZAMAN İÇİNDEKİ DEĞİŞİMİ
# ==========================================

daily_power = hourly["Global_active_power"].resample("D").mean()

plt.figure(figsize=(12, 5))

plt.plot(
    daily_power.index,
    daily_power.values,
    linewidth=0.8
)

plt.xlabel("Tarih")
plt.ylabel("Ortalama Aktif Güç (kW)")
plt.title("Günlük Ortalama Elektrik Tüketiminin Zaman İçindeki Değişimi")

plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    "gunluk_ortalama_tuketim.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "gunluk_ortalama_tuketim.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
print(hourly.columns.tolist())

In [ ]:
hourly["Energy_kWh"] = hourly["Global_active_power"] * 1.0

In [ ]:
hourly[
    ["Global_active_power", "Energy_kWh"]
].head()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    hourly["Energy_kWh"],
    bins=60,
    edgecolor="black",
    alpha=0.7
)

plt.xlabel("Saatlik Enerji Tüketimi (kWh)")
plt.ylabel("Frekans")
plt.title("Saatlik Enerji Tüketimi Dağılımı")

plt.tight_layout()

plt.savefig(
    "saatlik_enerji_dagilimi.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "saatlik_enerji_dagilimi.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
print("=" * 60)
print("SAATLİK VERİ SETİ KONTROLÜ")
print("=" * 60)

print("Satır sayısı:", len(hourly))
print("Başlangıç tarihi:", hourly.index.min())
print("Bitiş tarihi:", hourly.index.max())

print("\nEksik değer sayıları:")
print(hourly.isnull().sum())

print("\nGlobal Active Power:")
print("Minimum:", hourly["Global_active_power"].min())
print("Maksimum:", hourly["Global_active_power"].max())
print("Ortalama:", hourly["Global_active_power"].mean())
print("Standart sapma:", hourly["Global_active_power"].std())

print("\nNegatif değer sayısı:")
print((hourly["Global_active_power"] < 0).sum())

In [ ]:
# ==========================================
# 28. HEDEF DEĞİŞKENİN OLUŞTURULMASI
# ==========================================

hourly["Energy_kWh"] = hourly["Global_active_power"] * 1.0

print(hourly[["Global_active_power", "Energy_kWh"]].head())

print("\nEnergy_kWh sütunu mevcut mu?:",
      "Energy_kWh" in hourly.columns)

In [ ]:
# ==========================================
# 29. SAATLİK ENERJİ TÜKETİMİ DAĞILIMI
# ==========================================

plt.figure(figsize=(9, 5))

plt.hist(
    hourly["Energy_kWh"],
    bins=60,
    edgecolor="black",
    alpha=0.7
)

plt.xlabel("Saatlik Enerji Tüketimi (kWh)")
plt.ylabel("Frekans")
plt.title("Saatlik Enerji Tüketimi Dağılımı")

plt.grid(axis="y", alpha=0.2)
plt.tight_layout()

plt.savefig(
    "saatlik_enerji_dagilimi.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "saatlik_enerji_dagilimi.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 30. IQR İLE AYKIRI DEĞER ANALİZİ
# ==========================================

Q1 = hourly["Global_active_power"].quantile(0.25)
Q3 = hourly["Global_active_power"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_mask = (
    (hourly["Global_active_power"] < lower_bound) |
    (hourly["Global_active_power"] > upper_bound)
)

outlier_count = outlier_mask.sum()
outlier_percent = outlier_count / len(hourly) * 100

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)

print("\nAlt sınır:", lower_bound)
print("Üst sınır:", upper_bound)

print("\nIQR'ye göre aykırı değer sayısı:", outlier_count)
print(f"Aykırı değer oranı: %{outlier_percent:.2f}")

In [ ]:
# ==========================================
# 31. TÜKETİM YÜZDELİKLERİ
# ==========================================

quantiles = hourly["Global_active_power"].quantile([
    0.00,
    0.01,
    0.05,
    0.25,
    0.50,
    0.75,
    0.95,
    0.99,
    1.00
])

print(quantiles)

In [ ]:
# ==========================================
# 32. BOX PLOT
# ==========================================

plt.figure(figsize=(8, 4))

plt.boxplot(
    hourly["Global_active_power"],
    vert=False
)

plt.xlabel("Saatlik Ortalama Aktif Güç (kW)")
plt.title("Saatlik Aktif Güç Dağılımı")

plt.tight_layout()

plt.savefig(
    "aktif_guc_boxplot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "aktif_guc_boxplot.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 34. ZAMAN ÖZELLİKLERİ
# ==========================================

hourly["hour"] = hourly.index.hour
hourly["day_of_week"] = hourly.index.dayofweek
hourly["month"] = hourly.index.month

hourly["is_weekend"] = (
    hourly.index.dayofweek >= 5
).astype(int)

hourly[
    ["hour", "day_of_week", "month", "is_weekend"]
].head()

In [ ]:
# ==========================================
# 35. CYCLICAL TIME ENCODING
# ==========================================

hourly["hour_sin"] = np.sin(
    2 * np.pi * hourly["hour"] / 24
)

hourly["hour_cos"] = np.cos(
    2 * np.pi * hourly["hour"] / 24
)

hourly["dow_sin"] = np.sin(
    2 * np.pi * hourly["day_of_week"] / 7
)

hourly["dow_cos"] = np.cos(
    2 * np.pi * hourly["day_of_week"] / 7
)

hourly[
    [
        "hour",
        "hour_sin",
        "hour_cos",
        "day_of_week",
        "dow_sin",
        "dow_cos"
    ]
].head()

In [ ]:
# ==========================================
# 36. SAATE GÖRE ORTALAMA TÜKETİM
# ==========================================

hourly_profile = hourly.groupby("hour")[
    "Global_active_power"
].mean()

plt.figure(figsize=(9, 5))

plt.plot(
    hourly_profile.index,
    hourly_profile.values,
    marker="o"
)

plt.xticks(range(0, 24))
plt.xlabel("Saat")
plt.ylabel("Ortalama Aktif Güç (kW)")
plt.title("Günün Saatine Göre Ortalama Elektrik Tüketimi")

plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    "saatlik_tuketim_profili.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "saatlik_tuketim_profili.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 37. HAFTANIN GÜNLERİNE GÖRE TÜKETİM
# ==========================================

day_profile = hourly.groupby("day_of_week")[
    "Global_active_power"
].mean()

day_names = [
    "Pzt",
    "Sal",
    "Çar",
    "Per",
    "Cum",
    "Cmt",
    "Paz"
]

plt.figure(figsize=(8, 5))

plt.bar(
    day_names,
    day_profile.values
)

plt.xlabel("Haftanın Günü")
plt.ylabel("Ortalama Aktif Güç (kW)")
plt.title("Haftanın Günlerine Göre Ortalama Elektrik Tüketimi")

plt.tight_layout()

plt.savefig(
    "haftalik_tuketim_profili.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "haftalik_tuketim_profili.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 38. KORELASYON ANALİZİ
# ==========================================

correlation_columns = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

correlation_matrix = hourly[
    correlation_columns
].corr()

correlation_matrix

In [ ]:
import seaborn as sns

plt.figure(figsize=(9, 7))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    square=True
)

plt.title("Elektriksel Değişkenler Arasındaki Korelasyon")
plt.tight_layout()

plt.savefig(
    "korelasyon_matrisi.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "korelasyon_matrisi.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 39. FEATURE VE TARGET TANIMLARI
# ==========================================

FEATURES = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "is_weekend"
]

TARGET = "Energy_kWh"

print("Özellik sayısı:", len(FEATURES))
print("\nÖzellikler:")

for i, feature in enumerate(FEATURES, start=1):
    print(i, feature)

print("\nHedef:", TARGET)

In [ ]:
# ==========================================
# 40. TRAIN / VALIDATION / TEST
# ==========================================

n = len(hourly)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = hourly.iloc[:train_end].copy()
val_df   = hourly.iloc[train_end:val_end].copy()
test_df  = hourly.iloc[val_end:].copy()

print("Toplam:", len(hourly))

print("\nTrain:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
# ==========================================
# 41. SPLIT TARİHLERİ
# ==========================================

print(
    "TRAIN:",
    train_df.index.min(),
    "→",
    train_df.index.max()
)

print(
    "\nVALIDATION:",
    val_df.index.min(),
    "→",
    val_df.index.max()
)

print(
    "\nTEST:",
    test_df.index.min(),
    "→",
    test_df.index.max()
)

In [ ]:
# ==========================================
# 42. VERİ BÖLÜNMESİ GRAFİĞİ
# ==========================================

plt.figure(figsize=(13, 5))

plt.plot(
    train_df.index,
    train_df["Global_active_power"],
    label="Eğitim",
    linewidth=0.7
)

plt.plot(
    val_df.index,
    val_df["Global_active_power"],
    label="Doğrulama",
    linewidth=0.7
)

plt.plot(
    test_df.index,
    test_df["Global_active_power"],
    label="Test",
    linewidth=0.7
)

plt.xlabel("Tarih")
plt.ylabel("Aktif Güç (kW)")
plt.title("Zamansal Eğitim, Doğrulama ve Test Ayrımı")

plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()

plt.savefig(
    "train_validation_test_split.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "train_validation_test_split.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 43. NORMALİZASYON
# ==========================================

from sklearn.preprocessing import MinMaxScaler

feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

In [ ]:
feature_scaler.fit(
    train_df[FEATURES]
)

target_scaler.fit(
    train_df[[TARGET]]
)

In [ ]:
X_train_scaled = feature_scaler.transform(
    train_df[FEATURES]
)

X_val_scaled = feature_scaler.transform(
    val_df[FEATURES]
)

X_test_scaled = feature_scaler.transform(
    test_df[FEATURES]
)


y_train_scaled = target_scaler.transform(
    train_df[[TARGET]]
)

y_val_scaled = target_scaler.transform(
    val_df[[TARGET]]
)

y_test_scaled = target_scaler.transform(
    test_df[[TARGET]]
)

In [ ]:
print("X train:", X_train_scaled.shape)
print("X val:", X_val_scaled.shape)
print("X test:", X_test_scaled.shape)

print()

print("y train:", y_train_scaled.shape)
print("y val:", y_val_scaled.shape)
print("y test:", y_test_scaled.shape)

In [ ]:
SEQ_LEN = 24

In [ ]:
# ==========================================
# 44. SEQUENCE UZUNLUĞU
# ==========================================

SEQ_LEN = 24

print(
    f"Model geçmiş {SEQ_LEN} saati kullanarak "
    "bir sonraki saati tahmin edecek."
)

In [ ]:
# ==========================================
# 45. TRAIN SEQUENCE OLUŞTURMA
# ==========================================

def create_sequences(X, y, seq_len):

    X_seq = []
    y_seq = []

    for i in range(seq_len, len(X)):

        X_seq.append(
            X[i-seq_len:i]
        )

        y_seq.append(
            y[i]
        )

    return (
        np.asarray(X_seq, dtype=np.float32),
        np.asarray(y_seq, dtype=np.float32)
    )

In [ ]:
X_train, y_train = create_sequences(
    X_train_scaled,
    y_train_scaled,
    SEQ_LEN
)

In [ ]:
# ==========================================
# 46. VALIDATION/TEST SEQUENCE
# ==========================================

def create_eval_sequences(
    previous_X,
    current_X,
    current_y,
    seq_len
):

    combined_X = np.concatenate(
        [
            previous_X[-seq_len:],
            current_X
        ],
        axis=0
    )

    X_seq = []
    y_seq = []

    for i in range(seq_len, len(combined_X)):

        X_seq.append(
            combined_X[i-seq_len:i]
        )

        current_index = i - seq_len

        y_seq.append(
            current_y[current_index]
        )

    return (
        np.asarray(X_seq, dtype=np.float32),
        np.asarray(y_seq, dtype=np.float32)
    )

In [ ]:

X_val, y_val = create_eval_sequences(
    X_train_scaled,
    X_val_scaled,
    y_val_scaled,
    SEQ_LEN
)


In [ ]:
X_test, y_test = create_eval_sequences(
    X_val_scaled,
    X_test_scaled,
    y_test_scaled,
    SEQ_LEN
)

In [ ]:
# ==========================================
# 47. MODEL GİRİŞ BOYUTLARI
# ==========================================

print("=" * 60)

print("TRAIN")
print("X:", X_train.shape)
print("y:", y_train.shape)

print("=" * 60)

print("VALIDATION")
print("X:", X_val.shape)
print("y:", y_val.shape)

print("=" * 60)

print("TEST")
print("X:", X_test.shape)
print("y:", y_test.shape)

print("=" * 60)

print("\nBir model girdisinin boyutu:")
print(X_train[0].shape)

In [ ]:
# ==========================================
# 48. SON VERİ KALİTE KONTROLÜ
# ==========================================

datasets = {
    "X_train": X_train,
    "y_train": y_train,
    "X_val": X_val,
    "y_val": y_val,
    "X_test": X_test,
    "y_test": y_test
}

for name, data in datasets.items():

    print(
        name,
        "| NaN:",
        np.isnan(data).sum(),
        "| Inf:",
        np.isinf(data).sum()
    )

In [ ]:
# ==========================================
# 49. ÖN İŞLEME ÖZET TABLOSU
# ==========================================

preprocessing_summary = pd.DataFrame({

    "Aşama": [
        "Ham veri",
        "Saatlik yeniden örnekleme sonrası",
        "Eğitim kümesi",
        "Doğrulama kümesi",
        "Test kümesi",
        "Sequence sonrası eğitim örneği",
        "Sequence sonrası doğrulama örneği",
        "Sequence sonrası test örneği"
    ],

    "Örnek Sayısı": [
        len(df),
        len(hourly),
        len(train_df),
        len(val_df),
        len(test_df),
        len(X_train),
        len(X_val),
        len(X_test)
    ]
})

preprocessing_summary

In [ ]:
# ==========================================
# 50. PYTORCH VE GPU KONTROLÜ
# ==========================================

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

print("PyTorch sürümü:", torch.__version__)
print("CUDA kullanılabilir mi?:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Kullanılan cihaz:", device)

In [ ]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

In [ ]:
# ==========================================
# 52. NUMPY -> PYTORCH TENSOR
# ==========================================

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

print("Train tensor:", X_train_tensor.shape)
print("Target tensor:", y_train_tensor.shape)

In [ ]:
# ==========================================
# 53. DATALOADER
# ==========================================

BATCH_SIZE = 256

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Train batch sayısı:", len(train_loader))
print("Validation batch sayısı:", len(val_loader))
print("Test batch sayısı:", len(test_loader))

In [ ]:
# ==========================================
# 54. ORTAK DENEY PARAMETRELERİ
# ==========================================

INPUT_SIZE = X_train.shape[2]

HIDDEN_SIZE = 64
LEARNING_RATE = 0.001
MAX_EPOCHS = 100
PATIENCE = 10

print("Girdi özellik sayısı:", INPUT_SIZE)
print("Sequence uzunluğu:", SEQ_LEN)
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Max epoch:", MAX_EPOCHS)
print("Early stopping patience:", PATIENCE)

In [ ]:
# ==========================================
# 55. MLP MODELİ
# ==========================================

class MLPModel(nn.Module):

    def __init__(self, seq_len, input_size):

        super().__init__()

        flattened_size = seq_len * input_size

        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flattened_size, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)


mlp_model = MLPModel(
    seq_len=SEQ_LEN,
    input_size=INPUT_SIZE
).to(device)

print(mlp_model)

In [ ]:
def count_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


print(
    "MLP toplam parametre:",
    count_parameters(mlp_model)
)

In [ ]:
# ==========================================
# 56. ORTAK EĞİTİM FONKSİYONU
# ==========================================

def train_model(
    model,
    train_loader,
    val_loader,
    learning_rate=0.001,
    max_epochs=100,
    patience=10
):

    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    train_losses = []
    val_losses = []

    best_val_loss = float("inf")
    best_state = None

    patience_counter = 0

    for epoch in range(max_epochs):

        # ------------------
        # TRAIN
        # ------------------

        model.train()

        train_loss_sum = 0.0

        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            predictions = model(X_batch)

            loss = criterion(
                predictions,
                y_batch
            )

            loss.backward()

            optimizer.step()

            train_loss_sum += (
                loss.item() * X_batch.size(0)
            )

        train_loss = (
            train_loss_sum /
            len(train_loader.dataset)
        )

        # ------------------
        # VALIDATION
        # ------------------

        model.eval()

        val_loss_sum = 0.0

        with torch.no_grad():

            for X_batch, y_batch in val_loader:

                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                predictions = model(X_batch)

                loss = criterion(
                    predictions,
                    y_batch
                )

                val_loss_sum += (
                    loss.item() * X_batch.size(0)
                )

        val_loss = (
            val_loss_sum /
            len(val_loader.dataset)
        )

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        print(
            f"Epoch {epoch+1:03d} | "
            f"Train Loss: {train_loss:.6f} | "
            f"Val Loss: {val_loss:.6f}"
        )

        # ------------------
        # EARLY STOPPING
        # ------------------

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            best_state = {
                key: value.cpu().clone()
                for key, value
                in model.state_dict().items()
            }

            patience_counter = 0

        else:

            patience_counter += 1

        if patience_counter >= patience:

            print(
                f"\nEarly stopping: "
                f"{epoch+1}. epoch"
            )

            break

    # En iyi validation ağırlıklarını geri yükle
    model.load_state_dict(best_state)

    model.to(device)

    return (
        model,
        train_losses,
        val_losses
    )

In [ ]:
# ==========================================
# 57. MLP EĞİTİMİ
# ==========================================

mlp_model, mlp_train_losses, mlp_val_losses = train_model(
    mlp_model,
    train_loader,
    val_loader,
    learning_rate=LEARNING_RATE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE
)

In [ ]:
# ==========================================
# 58. MLP LOSS GRAFİĞİ
# ==========================================

best_epoch = np.argmin(mlp_val_losses) + 1
best_val_loss = np.min(mlp_val_losses)

print("En iyi epoch:", best_epoch)
print("En iyi Validation Loss:", best_val_loss)

plt.figure(figsize=(9, 5))

plt.plot(
    range(1, len(mlp_train_losses) + 1),
    mlp_train_losses,
    label="Eğitim Kaybı"
)

plt.plot(
    range(1, len(mlp_val_losses) + 1),
    mlp_val_losses,
    label="Doğrulama Kaybı"
)

plt.axvline(
    best_epoch,
    linestyle="--",
    alpha=0.7,
    label=f"En İyi Epoch = {best_epoch}"
)

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("MLP - Eğitim ve Doğrulama Kayıp Eğrileri")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    "MLP_loss_egrisi.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "MLP_loss_egrisi.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 59. TEST SETİ TAHMİNLERİ
# ==========================================

def predict_model(model, data_loader):

    model.eval()

    predictions = []
    actuals = []

    with torch.no_grad():

        for X_batch, y_batch in data_loader:

            X_batch = X_batch.to(device)

            output = model(X_batch)

            predictions.append(
                output.cpu().numpy()
            )

            actuals.append(
                y_batch.numpy()
            )

    predictions = np.concatenate(
        predictions,
        axis=0
    )

    actuals = np.concatenate(
        actuals,
        axis=0
    )

    return predictions, actuals


mlp_pred_scaled, mlp_true_scaled = predict_model(
    mlp_model,
    test_loader
)

print("Tahmin boyutu:", mlp_pred_scaled.shape)
print("Gerçek boyutu:", mlp_true_scaled.shape)

In [ ]:
# ==========================================
# 60. GERÇEK ÖLÇEĞE GERİ DÖNÜŞÜM
# ==========================================

mlp_pred = target_scaler.inverse_transform(
    mlp_pred_scaled
).flatten()

mlp_true = target_scaler.inverse_transform(
    mlp_true_scaled
).flatten()

print("İlk 10 gerçek değer:")
print(mlp_true[:10])

print("\nİlk 10 tahmin:")
print(mlp_pred[:10])

In [ ]:
# ==========================================
# 60. GERÇEK ÖLÇEĞE GERİ DÖNÜŞÜM
# ==========================================

mlp_pred = target_scaler.inverse_transform(
    mlp_pred_scaled
).flatten()

mlp_true = target_scaler.inverse_transform(
    mlp_true_scaled
).flatten()

print("İlk 10 gerçek değer:")
print(mlp_true[:10])

print("\nİlk 10 tahmin:")
print(mlp_pred[:10])

In [ ]:
# ==========================================
# 62. MODEL SONUÇ TABLOSU
# ==========================================

results = []

results.append({
    "Model": "MLP",
    "MAE": mlp_mae,
    "RMSE": mlp_rmse,
    "R2": mlp_r2,
    "MAPE": mlp_mape,
    "SMAPE": mlp_smape,
    "Parameters": count_parameters(mlp_model),
    "Best_Epoch": best_epoch
})

results_df = pd.DataFrame(results)

results_df

In [ ]:
# ==========================================
# 59. TEST SETİ TAHMİNLERİ
# ==========================================

def predict_model(model, data_loader):

    model.eval()

    predictions = []
    actuals = []

    with torch.no_grad():

        for X_batch, y_batch in data_loader:

            X_batch = X_batch.to(device)

            output = model(X_batch)

            predictions.append(
                output.cpu().numpy()
            )

            actuals.append(
                y_batch.numpy()
            )

    predictions = np.concatenate(predictions, axis=0)
    actuals = np.concatenate(actuals, axis=0)

    return predictions, actuals


mlp_pred_scaled, mlp_true_scaled = predict_model(
    mlp_model,
    test_loader
)

print("Tahmin boyutu:", mlp_pred_scaled.shape)
print("Gerçek boyutu:", mlp_true_scaled.shape)

In [ ]:
# ==========================================
# 60. NORMALİZASYONU GERİ AL
# ==========================================

mlp_pred = target_scaler.inverse_transform(
    mlp_pred_scaled
).flatten()

mlp_true = target_scaler.inverse_transform(
    mlp_true_scaled
).flatten()

print("İlk 10 gerçek:")
print(mlp_true[:10])

print("\nİlk 10 tahmin:")
print(mlp_pred[:10])

In [ ]:
# ==========================================
# 61. MLP PERFORMANS METRİKLERİ
# ==========================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

def calculate_metrics(y_true, y_pred):

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    epsilon = 1e-8

    mape = np.mean(
        np.abs(
            (y_true - y_pred) /
            np.maximum(np.abs(y_true), epsilon)
        )
    ) * 100

    smape = np.mean(
        2 * np.abs(y_pred - y_true) /
        (
            np.abs(y_true) +
            np.abs(y_pred) +
            epsilon
        )
    ) * 100

    return mae, rmse, r2, mape, smape


mlp_mae, mlp_rmse, mlp_r2, mlp_mape, mlp_smape = calculate_metrics(
    mlp_true,
    mlp_pred
)

print("=" * 50)
print("MLP TEST SONUÇLARI")
print("=" * 50)

print(f"MAE   : {mlp_mae:.6f}")
print(f"RMSE  : {mlp_rmse:.6f}")
print(f"R²    : {mlp_r2:.6f}")
print(f"MAPE  : {mlp_mape:.3f}%")
print(f"SMAPE : {mlp_smape:.3f}%")

In [ ]:
# ==========================================
# 62. MODEL SONUÇ TABLOSU
# ==========================================

results = []

results.append({
    "Model": "MLP",
    "MAE": mlp_mae,
    "RMSE": mlp_rmse,
    "R2": mlp_r2,
    "MAPE": mlp_mape,
    "SMAPE": mlp_smape,
    "Parameters": count_parameters(mlp_model),
    "Best_Epoch": best_epoch
})

results_df = pd.DataFrame(results)

results_df

In [ ]:
# ==========================================
# 63. MLP GERÇEK - TAHMİN GRAFİĞİ
# ==========================================

N_PLOT = 200

plt.figure(figsize=(12, 5))

plt.plot(
    mlp_true[:N_PLOT],
    label="Gerçek",
    linewidth=2
)

plt.plot(
    mlp_pred[:N_PLOT],
    label="MLP Tahmini",
    linestyle="--",
    linewidth=1.7
)

plt.xlabel("Test Zaman Adımı (Saat)")
plt.ylabel("Saatlik Tüketim (kWh)")
plt.title("MLP - Gerçek ve Tahmin Edilen Saatlik Tüketim")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    "MLP_prediction_plot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "MLP_prediction_plot.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 64. MLP RESIDUAL PLOT
# ==========================================

mlp_residuals = mlp_true - mlp_pred

plt.figure(figsize=(12, 5))

plt.scatter(
    np.arange(len(mlp_residuals)),
    mlp_residuals,
    s=8,
    alpha=0.5
)

plt.axhline(
    y=0,
    linestyle="--",
    linewidth=1.5
)

plt.xlabel("Test Örneği")
plt.ylabel("Residual (Gerçek - Tahmin)")
plt.title("MLP - Residual Analizi")

plt.grid(alpha=0.2)
plt.tight_layout()

plt.savefig(
    "MLP_residual_plot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "MLP_residual_plot.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 65. MLP HATA DAĞILIMI
# ==========================================

plt.figure(figsize=(9, 5))

plt.hist(
    mlp_residuals,
    bins=50,
    edgecolor="black",
    alpha=0.7
)

plt.axvline(
    0,
    linestyle="--",
    linewidth=1.5
)

plt.xlabel("Residual (Gerçek - Tahmin)")
plt.ylabel("Frekans")
plt.title("MLP - Tahmin Hatalarının Dağılımı")

plt.grid(axis="y", alpha=0.2)
plt.tight_layout()

plt.savefig(
    "MLP_hata_dagilimi.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "MLP_hata_dagilimi.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 66. MLP BIAS
# ==========================================

mlp_bias = np.mean(
    mlp_pred - mlp_true
)

print(
    f"MLP Ortalama Tahmin Bias: {mlp_bias:.6f} kWh"
)

In [ ]:
# ==========================================
# 67. MLP MODEL BOYUTU
# ==========================================

torch.save(
    mlp_model.state_dict(),
    "mlp_model.pth"
)

model_size_bytes = os.path.getsize(
    "mlp_model.pth"
)

model_size_kb = (
    model_size_bytes / 1024
)

print(
    f"MLP model dosya boyutu: "
    f"{model_size_kb:.2f} KB"
)

In [ ]:
# FP32 = parametre başına 4 byte

mlp_parameters = count_parameters(
    mlp_model
)

mlp_fp32_memory_kb = (
    mlp_parameters * 4 / 1024
)

mlp_int16_memory_kb = (
    mlp_parameters * 2 / 1024
)

mlp_int8_memory_kb = (
    mlp_parameters * 1 / 1024
)

print(
    f"Parametre sayısı : {mlp_parameters:,}"
)

print(
    f"FP32 teorik bellek : "
    f"{mlp_fp32_memory_kb:.2f} KB"
)

print(
    f"INT16 teorik bellek: "
    f"{mlp_int16_memory_kb:.2f} KB"
)

print(
    f"INT8 teorik bellek : "
    f"{mlp_int8_memory_kb:.2f} KB"
)

In [ ]:
# ==========================================
# 68. MLP CPU INFERENCE SÜRESİ
# ==========================================

import copy
import time

mlp_cpu = copy.deepcopy(
    mlp_model
).cpu()

mlp_cpu.eval()

sample = X_test_tensor[0:1]

# Warm-up
with torch.no_grad():
    for _ in range(100):
        _ = mlp_cpu(sample)

N_RUNS = 1000

start = time.perf_counter()

with torch.no_grad():

    for _ in range(N_RUNS):
        _ = mlp_cpu(sample)

end = time.perf_counter()

avg_time_seconds = (
    (end - start) / N_RUNS
)

avg_time_ms = (
    avg_time_seconds * 1000
)

avg_time_us = (
    avg_time_seconds * 1_000_000
)

print(
    f"Ortalama CPU inference: "
    f"{avg_time_ms:.6f} ms"
)

print(
    f"Ortalama CPU inference: "
    f"{avg_time_us:.2f} µs"
)

In [ ]:
print("=" * 60)
print("MLP ÖZET")
print("=" * 60)

print(f"En iyi epoch       : {best_epoch}")
print(f"Parametre          : {mlp_parameters:,}")
print(f"MAE                : {mlp_mae:.6f}")
print(f"RMSE               : {mlp_rmse:.6f}")
print(f"R²                 : {mlp_r2:.6f}")
print(f"MAPE               : {mlp_mape:.3f}%")
print(f"SMAPE              : {mlp_smape:.3f}%")
print(f"Bias               : {mlp_bias:.6f}")
print(f"FP32 ağırlık       : {mlp_fp32_memory_kb:.2f} KB")
print(f"INT16 ağırlık      : {mlp_int16_memory_kb:.2f} KB")
print(f"INT8 ağırlık       : {mlp_int8_memory_kb:.2f} KB")
print(f"CPU inference      : {avg_time_us:.2f} µs")

In [ ]:
# ==========================================
# 70. NAIVE / PERSISTENCE BASELINE
# ==========================================

# Her test sequence'inin son saatindeki
# Global_active_power değerini al
naive_scaled = X_test[:, -1, 0].reshape(-1, 1)

# Global_active_power ile hedefimiz sayısal olarak aynı ölçeğe sahip
naive_pred = target_scaler.inverse_transform(
    naive_scaled
).flatten()

naive_true = mlp_true.copy()

(
    naive_mae,
    naive_rmse,
    naive_r2,
    naive_mape,
    naive_smape
) = calculate_metrics(
    naive_true,
    naive_pred
)

print("=" * 50)
print("NAIVE / PERSISTENCE BASELINE")
print("=" * 50)

print(f"MAE   : {naive_mae:.6f}")
print(f"RMSE  : {naive_rmse:.6f}")
print(f"R²    : {naive_r2:.6f}")
print(f"MAPE  : {naive_mape:.3f}%")
print(f"SMAPE : {naive_smape:.3f}%")

In [ ]:
# ==========================================
# 71. 24 SAATLİK SEASONAL NAIVE BASELINE
# ==========================================

seasonal_scaled = X_test[:, 0, 0].reshape(-1, 1)

seasonal_pred = target_scaler.inverse_transform(
    seasonal_scaled
).flatten()

(
    seasonal_mae,
    seasonal_rmse,
    seasonal_r2,
    seasonal_mape,
    seasonal_smape
) = calculate_metrics(
    mlp_true,
    seasonal_pred
)

print("=" * 50)
print("24 SAATLİK SEASONAL NAIVE BASELINE")
print("=" * 50)

print(f"MAE   : {seasonal_mae:.6f}")
print(f"RMSE  : {seasonal_rmse:.6f}")
print(f"R²    : {seasonal_r2:.6f}")
print(f"MAPE  : {seasonal_mape:.3f}%")
print(f"SMAPE : {seasonal_smape:.3f}%")

In [ ]:
# ==========================================
# 73. LSTM MODELİ
# ==========================================

class LSTMModel(nn.Module):

    def __init__(
        self,
        input_size,
        hidden_size=64
    ):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            1
        )

    def forward(self, x):

        lstm_out, _ = self.lstm(x)

        # Son zaman adımının çıktısı
        last_output = lstm_out[:, -1, :]

        prediction = self.fc(
            last_output
        )

        return prediction


lstm_model = LSTMModel(
    input_size=INPUT_SIZE,
    hidden_size=HIDDEN_SIZE
).to(device)

print(lstm_model)

print(
    "\nLSTM toplam parametre:",
    count_parameters(lstm_model)
)

In [ ]:
# ==========================================
# 74. LSTM EĞİTİMİ
# ==========================================

lstm_model, lstm_train_losses, lstm_val_losses = train_model(
    lstm_model,
    train_loader,
    val_loader,
    learning_rate=LEARNING_RATE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE
)

In [ ]:
# ==========================================
# 75. LSTM EN İYİ EPOCH
# ==========================================

lstm_best_epoch = (
    np.argmin(lstm_val_losses) + 1
)

lstm_best_val_loss = np.min(
    lstm_val_losses
)

print(
    "LSTM en iyi epoch:",
    lstm_best_epoch
)

print(
    "LSTM en iyi Validation Loss:",
    lstm_best_val_loss
)

In [ ]:
# ==========================================
# 76. LSTM LOSS GRAFİĞİ
# ==========================================

plt.figure(figsize=(9, 5))

plt.plot(
    range(1, len(lstm_train_losses) + 1),
    lstm_train_losses,
    label="Eğitim Kaybı"
)

plt.plot(
    range(1, len(lstm_val_losses) + 1),
    lstm_val_losses,
    label="Doğrulama Kaybı"
)

plt.axvline(
    lstm_best_epoch,
    linestyle="--",
    alpha=0.7,
    label=f"En İyi Epoch = {lstm_best_epoch}"
)

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")

plt.title(
    "LSTM - Eğitim ve Doğrulama Kayıp Eğrileri"
)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    "LSTM_loss_egrisi.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "LSTM_loss_egrisi.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 77. LSTM TEST TAHMİNLERİ
# ==========================================

lstm_pred_scaled, lstm_true_scaled = predict_model(
    lstm_model,
    test_loader
)

lstm_pred = target_scaler.inverse_transform(
    lstm_pred_scaled
).flatten()

lstm_true = target_scaler.inverse_transform(
    lstm_true_scaled
).flatten()

print(
    "Tahmin sayısı:",
    len(lstm_pred)
)

print(
    "Gerçek değer sayısı:",
    len(lstm_true)
)

In [ ]:
# ==========================================
# 78. LSTM TEST METRİKLERİ
# ==========================================

(
    lstm_mae,
    lstm_rmse,
    lstm_r2,
    lstm_mape,
    lstm_smape
) = calculate_metrics(
    lstm_true,
    lstm_pred
)

print("=" * 60)
print("LSTM TEST SONUÇLARI")
print("=" * 60)

print(f"MAE   : {lstm_mae:.6f}")
print(f"RMSE  : {lstm_rmse:.6f}")
print(f"R²    : {lstm_r2:.6f}")
print(f"MAPE  : {lstm_mape:.3f}%")
print(f"SMAPE : {lstm_smape:.3f}%")

In [ ]:
# ==========================================
# 79. LSTM'Yİ SONUÇ TABLOSUNA EKLE
# ==========================================

results.append({
    "Model": "LSTM",
    "MAE": lstm_mae,
    "RMSE": lstm_rmse,
    "R2": lstm_r2,
    "MAPE": lstm_mape,
    "SMAPE": lstm_smape,
    "Parameters": count_parameters(lstm_model),
    "Best_Epoch": lstm_best_epoch
})

results_df = pd.DataFrame(results)

results_df

In [ ]:
# ==========================================
# 80. LSTM GERÇEK - TAHMİN
# ==========================================

N_PLOT = 200

plt.figure(figsize=(12, 5))

plt.plot(
    lstm_true[:N_PLOT],
    label="Gerçek",
    linewidth=2
)

plt.plot(
    lstm_pred[:N_PLOT],
    label="LSTM Tahmini",
    linestyle="--",
    linewidth=1.7
)

plt.xlabel(
    "Test Zaman Adımı (Saat)"
)

plt.ylabel(
    "Ortalama Aktif Güç (kW)"
)

plt.title(
    "LSTM - Gerçek ve Tahmin Edilen Aktif Güç"
)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    "LSTM_prediction_plot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "LSTM_prediction_plot.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 81. LSTM RESIDUAL
# ==========================================

lstm_residuals = (
    lstm_true - lstm_pred
)

plt.figure(figsize=(12, 5))

plt.scatter(
    np.arange(len(lstm_residuals)),
    lstm_residuals,
    s=8,
    alpha=0.5
)

plt.axhline(
    y=0,
    linestyle="--",
    linewidth=1.5
)

plt.xlabel("Test Örneği")

plt.ylabel(
    "Residual (Gerçek - Tahmin)"
)

plt.title(
    "LSTM - Residual Analizi"
)

plt.grid(alpha=0.2)
plt.tight_layout()

plt.savefig(
    "LSTM_residual_plot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "LSTM_residual_plot.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 82. LSTM BIAS
# ==========================================

lstm_bias = np.mean(
    lstm_pred - lstm_true
)

print(
    f"LSTM Ortalama Tahmin Bias: "
    f"{lstm_bias:.6f} kW"
)

In [ ]:
# ==========================================
# 83. LSTM BELLEK
# ==========================================

lstm_parameters = count_parameters(
    lstm_model
)

lstm_fp32_memory_kb = (
    lstm_parameters * 4 / 1024
)

lstm_int16_memory_kb = (
    lstm_parameters * 2 / 1024
)

lstm_int8_memory_kb = (
    lstm_parameters / 1024
)

print(
    f"Parametre: {lstm_parameters:,}"
)

print(
    f"FP32 : {lstm_fp32_memory_kb:.2f} KB"
)

print(
    f"INT16: {lstm_int16_memory_kb:.2f} KB"
)

print(
    f"INT8 : {lstm_int8_memory_kb:.2f} KB"
)

In [ ]:
# ==========================================
# 84. LSTM CPU INFERENCE
# ==========================================

lstm_cpu = copy.deepcopy(
    lstm_model
).cpu()

lstm_cpu.eval()

sample = X_test_tensor[0:1]

with torch.no_grad():

    for _ in range(100):
        _ = lstm_cpu(sample)

N_RUNS = 1000

start = time.perf_counter()

with torch.no_grad():

    for _ in range(N_RUNS):
        _ = lstm_cpu(sample)

end = time.perf_counter()

lstm_avg_time_us = (
    (end - start) /
    N_RUNS *
    1_000_000
)

print(
    f"LSTM CPU inference: "
    f"{lstm_avg_time_us:.2f} µs"
)

In [ ]:
# ==========================================
# 85. GRU MODELİ
# ==========================================

class GRUModel(nn.Module):

    def __init__(
        self,
        input_size,
        hidden_size=64
    ):

        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            1
        )

    def forward(self, x):

        gru_out, _ = self.gru(x)

        # Son zaman adımındaki çıktı
        last_output = gru_out[:, -1, :]

        prediction = self.fc(last_output)

        return prediction


gru_model = GRUModel(
    input_size=INPUT_SIZE,
    hidden_size=HIDDEN_SIZE
).to(device)

print(gru_model)

print(
    "\nGRU toplam parametre:",
    count_parameters(gru_model)
)

In [ ]:
# ==========================================
# 86. GRU EĞİTİMİ
# ==========================================

gru_model, gru_train_losses, gru_val_losses = train_model(
    gru_model,
    train_loader,
    val_loader,
    learning_rate=LEARNING_RATE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE
)

In [ ]:
# ==========================================
# 87. GRU EN İYİ EPOCH
# ==========================================

gru_best_epoch = np.argmin(
    gru_val_losses
) + 1

gru_best_val_loss = np.min(
    gru_val_losses
)

print(
    "GRU en iyi epoch:",
    gru_best_epoch
)

print(
    "GRU en iyi Validation Loss:",
    gru_best_val_loss
)

In [ ]:
# ==========================================
# 88. GRU LOSS GRAFİĞİ
# ==========================================

plt.figure(figsize=(9, 5))

plt.plot(
    range(1, len(gru_train_losses) + 1),
    gru_train_losses,
    label="Eğitim Kaybı"
)

plt.plot(
    range(1, len(gru_val_losses) + 1),
    gru_val_losses,
    label="Doğrulama Kaybı"
)

plt.axvline(
    gru_best_epoch,
    linestyle="--",
    alpha=0.7,
    label=f"En İyi Epoch = {gru_best_epoch}"
)

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")

plt.title(
    "GRU - Eğitim ve Doğrulama Kayıp Eğrileri"
)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    "GRU_loss_egrisi.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "GRU_loss_egrisi.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 89. GRU TEST TAHMİNLERİ
# ==========================================

gru_pred_scaled, gru_true_scaled = predict_model(
    gru_model,
    test_loader
)

gru_pred = target_scaler.inverse_transform(
    gru_pred_scaled
).flatten()

gru_true = target_scaler.inverse_transform(
    gru_true_scaled
).flatten()

print("Tahmin:", gru_pred.shape)
print("Gerçek :", gru_true.shape)

In [ ]:
# ==========================================
# 90. GRU TEST METRİKLERİ
# ==========================================

(
    gru_mae,
    gru_rmse,
    gru_r2,
    gru_mape,
    gru_smape
) = calculate_metrics(
    gru_true,
    gru_pred
)

print("=" * 60)
print("GRU TEST SONUÇLARI")
print("=" * 60)

print(f"MAE   : {gru_mae:.6f}")
print(f"RMSE  : {gru_rmse:.6f}")
print(f"R²    : {gru_r2:.6f}")
print(f"MAPE  : {gru_mape:.3f}%")
print(f"SMAPE : {gru_smape:.3f}%")

In [ ]:
# ==========================================
# 91. GRU SONUÇ TABLOSU
# ==========================================

results.append({
    "Model": "GRU",
    "MAE": gru_mae,
    "RMSE": gru_rmse,
    "R2": gru_r2,
    "MAPE": gru_mape,
    "SMAPE": gru_smape,
    "Parameters": count_parameters(gru_model),
    "Best_Epoch": gru_best_epoch
})

results_df = pd.DataFrame(results)

results_df

In [ ]:
# ==========================================
# 92. GRU GERÇEK - TAHMİN
# ==========================================

N_PLOT = 200

plt.figure(figsize=(12, 5))

plt.plot(
    gru_true[:N_PLOT],
    label="Gerçek",
    linewidth=2
)

plt.plot(
    gru_pred[:N_PLOT],
    label="GRU Tahmini",
    linestyle="--",
    linewidth=1.7
)

plt.xlabel(
    "Test Zaman Adımı (Saat)"
)

plt.ylabel(
    "Ortalama Aktif Güç (kW)"
)

plt.title(
    "GRU - Gerçek ve Tahmin Edilen Aktif Güç"
)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    "GRU_prediction_plot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "GRU_prediction_plot.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 93. GRU RESIDUAL
# ==========================================

gru_residuals = (
    gru_true - gru_pred
)

plt.figure(figsize=(12, 5))

plt.scatter(
    np.arange(len(gru_residuals)),
    gru_residuals,
    s=8,
    alpha=0.5
)

plt.axhline(
    y=0,
    linestyle="--",
    linewidth=1.5
)

plt.xlabel("Test Örneği")

plt.ylabel(
    "Residual (Gerçek - Tahmin)"
)

plt.title(
    "GRU - Residual Analizi"
)

plt.grid(alpha=0.2)
plt.tight_layout()

plt.savefig(
    "GRU_residual_plot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "GRU_residual_plot.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 94. GRU HATA DAĞILIMI
# ==========================================

plt.figure(figsize=(9, 5))

plt.hist(
    gru_residuals,
    bins=50,
    edgecolor="black",
    alpha=0.7
)

plt.axvline(
    0,
    linestyle="--",
    linewidth=1.5
)

plt.xlabel(
    "Residual (Gerçek - Tahmin)"
)

plt.ylabel("Frekans")

plt.title(
    "GRU - Tahmin Hatalarının Dağılımı"
)

plt.grid(axis="y", alpha=0.2)
plt.tight_layout()

plt.savefig(
    "GRU_hata_dagilimi.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "GRU_hata_dagilimi.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 95. GRU BIAS + BELLEK
# ==========================================

gru_bias = np.mean(
    gru_pred - gru_true
)

gru_parameters = count_parameters(
    gru_model
)

gru_fp32_memory_kb = (
    gru_parameters * 4 / 1024
)

gru_int16_memory_kb = (
    gru_parameters * 2 / 1024
)

gru_int8_memory_kb = (
    gru_parameters / 1024
)

print(
    f"GRU Bias: {gru_bias:.6f} kW"
)

print(
    f"Parametre: {gru_parameters:,}"
)

print(
    f"FP32 : {gru_fp32_memory_kb:.2f} KB"
)

print(
    f"INT16: {gru_int16_memory_kb:.2f} KB"
)

print(
    f"INT8 : {gru_int8_memory_kb:.2f} KB"
)

In [ ]:
# ==========================================
# 96. GRU CPU INFERENCE
# ==========================================

gru_cpu = copy.deepcopy(
    gru_model
).cpu()

gru_cpu.eval()

sample = X_test_tensor[0:1]

with torch.no_grad():

    for _ in range(100):
        _ = gru_cpu(sample)

N_RUNS = 1000

start = time.perf_counter()

with torch.no_grad():

    for _ in range(N_RUNS):
        _ = gru_cpu(sample)

end = time.perf_counter()

gru_avg_time_us = (
    (end - start) /
    N_RUNS *
    1_000_000
)

print(
    f"GRU CPU inference: "
    f"{gru_avg_time_us:.2f} µs"
)

In [ ]:
# ==========================================
# 97. BiLSTM MODELİ
# ==========================================

class BiLSTMModel(nn.Module):

    def __init__(
        self,
        input_size,
        hidden_size=64
    ):

        super().__init__()

        self.bilstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # Bidirectional olduğu için çıktı boyutu 2*hidden_size
        self.fc = nn.Linear(
            hidden_size * 2,
            1
        )

    def forward(self, x):

        bilstm_out, _ = self.bilstm(x)

        # Son zaman adımındaki ileri+geri çıktı
        last_output = bilstm_out[:, -1, :]

        prediction = self.fc(last_output)

        return prediction


bilstm_model = BiLSTMModel(
    input_size=INPUT_SIZE,
    hidden_size=HIDDEN_SIZE
).to(device)

print(bilstm_model)

print(
    "\nBiLSTM toplam parametre:",
    count_parameters(bilstm_model)
)

In [ ]:
# ==========================================
# 98. BiLSTM EĞİTİMİ
# ==========================================

bilstm_model, bilstm_train_losses, bilstm_val_losses = train_model(
    bilstm_model,
    train_loader,
    val_loader,
    learning_rate=LEARNING_RATE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE
)

In [ ]:
# ==========================================
# 99. BiLSTM EN İYİ EPOCH
# ==========================================

bilstm_best_epoch = np.argmin(
    bilstm_val_losses
) + 1

bilstm_best_val_loss = np.min(
    bilstm_val_losses
)

print("BiLSTM en iyi epoch:", bilstm_best_epoch)
print("BiLSTM en iyi Validation Loss:", bilstm_best_val_loss)

In [ ]:
# ==========================================
# 100. BiLSTM LOSS GRAFİĞİ
# ==========================================

plt.figure(figsize=(9, 5))

plt.plot(
    range(1, len(bilstm_train_losses) + 1),
    bilstm_train_losses,
    label="Eğitim Kaybı"
)

plt.plot(
    range(1, len(bilstm_val_losses) + 1),
    bilstm_val_losses,
    label="Doğrulama Kaybı"
)

plt.axvline(
    bilstm_best_epoch,
    linestyle="--",
    alpha=0.7,
    label=f"En İyi Epoch = {bilstm_best_epoch}"
)

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("BiLSTM - Eğitim ve Doğrulama Kayıp Eğrileri")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    "BiLSTM_loss_egrisi_yeni.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "BiLSTM_loss_egrisi_yeni.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 101. BiLSTM TEST TAHMİNLERİ
# ==========================================

bilstm_pred_scaled, bilstm_true_scaled = predict_model(
    bilstm_model,
    test_loader
)

bilstm_pred = target_scaler.inverse_transform(
    bilstm_pred_scaled
).flatten()

bilstm_true = target_scaler.inverse_transform(
    bilstm_true_scaled
).flatten()

print("Tahmin:", bilstm_pred.shape)
print("Gerçek :", bilstm_true.shape)

In [ ]:
# ==========================================
# 102. BiLSTM TEST METRİKLERİ
# ==========================================

(
    bilstm_mae,
    bilstm_rmse,
    bilstm_r2,
    bilstm_mape,
    bilstm_smape
) = calculate_metrics(
    bilstm_true,
    bilstm_pred
)

print("=" * 60)
print("BiLSTM TEST SONUÇLARI")
print("=" * 60)

print(f"MAE   : {bilstm_mae:.6f}")
print(f"RMSE  : {bilstm_rmse:.6f}")
print(f"R²    : {bilstm_r2:.6f}")
print(f"MAPE  : {bilstm_mape:.3f}%")
print(f"SMAPE : {bilstm_smape:.3f}%")

In [ ]:
# ==========================================
# 103. BiLSTM'Yİ SONUÇ TABLOSUNA EKLE
# ==========================================

results.append({
    "Model": "BiLSTM",
    "MAE": bilstm_mae,
    "RMSE": bilstm_rmse,
    "R2": bilstm_r2,
    "MAPE": bilstm_mape,
    "SMAPE": bilstm_smape,
    "Parameters": count_parameters(bilstm_model),
    "Best_Epoch": bilstm_best_epoch
})

results_df = pd.DataFrame(results)

results_df

In [ ]:
# ==========================================
# 104. BiLSTM GERÇEK - TAHMİN
# ==========================================

N_PLOT = 200

plt.figure(figsize=(12, 5))

plt.plot(
    bilstm_true[:N_PLOT],
    label="Gerçek",
    linewidth=2
)

plt.plot(
    bilstm_pred[:N_PLOT],
    label="BiLSTM Tahmini",
    linestyle="--",
    linewidth=1.7
)

plt.xlabel("Test Zaman Adımı (Saat)")
plt.ylabel("Ortalama Aktif Güç (kW)")
plt.title("BiLSTM - Gerçek ve Tahmin Edilen Aktif Güç")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    "BiLSTM_prediction_plot_yeni.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "BiLSTM_prediction_plot_yeni.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 105. BiLSTM RESIDUAL
# ==========================================

bilstm_residuals = bilstm_true - bilstm_pred

plt.figure(figsize=(12, 5))

plt.scatter(
    np.arange(len(bilstm_residuals)),
    bilstm_residuals,
    s=8,
    alpha=0.5
)

plt.axhline(
    y=0,
    linestyle="--",
    linewidth=1.5
)

plt.xlabel("Test Örneği")
plt.ylabel("Residual (Gerçek - Tahmin)")
plt.title("BiLSTM - Residual Analizi")

plt.grid(alpha=0.2)
plt.tight_layout()

plt.savefig(
    "BiLSTM_residual_plot_yeni.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "BiLSTM_residual_plot_yeni.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 106. BiLSTM HATA DAĞILIMI
# ==========================================

plt.figure(figsize=(9, 5))

plt.hist(
    bilstm_residuals,
    bins=50,
    edgecolor="black",
    alpha=0.7
)

plt.axvline(
    0,
    linestyle="--",
    linewidth=1.5
)

plt.xlabel("Residual (Gerçek - Tahmin)")
plt.ylabel("Frekans")
plt.title("BiLSTM - Tahmin Hatalarının Dağılımı")

plt.grid(axis="y", alpha=0.2)
plt.tight_layout()

plt.savefig(
    "BiLSTM_hata_dagilimi_yeni.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "BiLSTM_hata_dagilimi_yeni.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 107. BiLSTM BIAS + BELLEK
# ==========================================

bilstm_bias = np.mean(
    bilstm_pred - bilstm_true
)

bilstm_parameters = count_parameters(
    bilstm_model
)

bilstm_fp32_memory_kb = (
    bilstm_parameters * 4 / 1024
)

bilstm_int16_memory_kb = (
    bilstm_parameters * 2 / 1024
)

bilstm_int8_memory_kb = (
    bilstm_parameters / 1024
)

print(f"BiLSTM Bias: {bilstm_bias:.6f} kW")
print(f"Parametre: {bilstm_parameters:,}")
print(f"FP32 : {bilstm_fp32_memory_kb:.2f} KB")
print(f"INT16: {bilstm_int16_memory_kb:.2f} KB")
print(f"INT8 : {bilstm_int8_memory_kb:.2f} KB")

In [ ]:
# ==========================================
# 108. BiLSTM CPU INFERENCE
# ==========================================

bilstm_cpu = copy.deepcopy(
    bilstm_model
).cpu()

bilstm_cpu.eval()

sample = X_test_tensor[0:1]

with torch.no_grad():
    for _ in range(100):
        _ = bilstm_cpu(sample)

N_RUNS = 1000

start = time.perf_counter()

with torch.no_grad():
    for _ in range(N_RUNS):
        _ = bilstm_cpu(sample)

end = time.perf_counter()

bilstm_avg_time_us = (
    (end - start) / N_RUNS * 1_000_000
)

print(
    f"BiLSTM CPU inference: "
    f"{bilstm_avg_time_us:.2f} µs"
)

In [ ]:
# ==========================================
# 109. DÜZELTİLMİŞ BiLSTM MODELİ
# ==========================================

class BiLSTMModelCorrected(nn.Module):

    def __init__(
        self,
        input_size,
        hidden_size=64
    ):

        super().__init__()

        self.bilstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(
            hidden_size * 2,
            1
        )

    def forward(self, x):

        _, (h_n, _) = self.bilstm(x)

        # h_n şekli:
        # [2, batch, hidden_size]
        #
        # h_n[0] = ileri yönün son hidden state'i
        # h_n[1] = geri yönün son hidden state'i

        forward_hidden = h_n[0]
        backward_hidden = h_n[1]

        combined = torch.cat(
            (forward_hidden, backward_hidden),
            dim=1
        )

        prediction = self.fc(combined)

        return prediction


bilstm_model = BiLSTMModelCorrected(
    input_size=INPUT_SIZE,
    hidden_size=HIDDEN_SIZE
).to(device)

print(bilstm_model)

print(
    "\nBiLSTM toplam parametre:",
    count_parameters(bilstm_model)
)

In [ ]:
# ==========================================
# 110. DÜZELTİLMİŞ BiLSTM EĞİTİMİ
# ==========================================

bilstm_model, bilstm_train_losses, bilstm_val_losses = train_model(
    bilstm_model,
    train_loader,
    val_loader,
    learning_rate=LEARNING_RATE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE
)

In [ ]:
bilstm_best_epoch = np.argmin(
    bilstm_val_losses
) + 1

bilstm_best_val_loss = np.min(
    bilstm_val_losses
)

print(
    "BiLSTM en iyi epoch:",
    bilstm_best_epoch
)

print(
    "BiLSTM en iyi validation loss:",
    bilstm_best_val_loss
)

In [ ]:
bilstm_pred_scaled, bilstm_true_scaled = predict_model(
    bilstm_model,
    test_loader
)

bilstm_pred = target_scaler.inverse_transform(
    bilstm_pred_scaled
).flatten()

bilstm_true = target_scaler.inverse_transform(
    bilstm_true_scaled
).flatten()

In [ ]:
(
    bilstm_mae,
    bilstm_rmse,
    bilstm_r2,
    bilstm_mape,
    bilstm_smape
) = calculate_metrics(
    bilstm_true,
    bilstm_pred
)

print("=" * 60)
print("DÜZELTİLMİŞ BiLSTM TEST SONUÇLARI")
print("=" * 60)

print(f"MAE   : {bilstm_mae:.6f}")
print(f"RMSE  : {bilstm_rmse:.6f}")
print(f"R²    : {bilstm_r2:.6f}")
print(f"MAPE  : {bilstm_mape:.3f}%")
print(f"SMAPE : {bilstm_smape:.3f}%")

In [ ]:
bilstm_bias = np.mean(
    bilstm_pred - bilstm_true
)

print(
    f"BiLSTM Bias: {bilstm_bias:.6f} kW"
)

In [ ]:
# ==========================================
# 115. BiLSTM SONUCUNU GÜNCELLE
# ==========================================

# Eski BiLSTM varsa sonuç listesinden kaldır
results = [
    row for row in results
    if row["Model"] != "BiLSTM"
]

# Düzeltilmiş BiLSTM sonucunu ekle
results.append({
    "Model": "BiLSTM",
    "MAE": bilstm_mae,
    "RMSE": bilstm_rmse,
    "R2": bilstm_r2,
    "MAPE": bilstm_mape,
    "SMAPE": bilstm_smape,
    "Parameters": count_parameters(bilstm_model),
    "Best_Epoch": bilstm_best_epoch
})

# DataFrame'i yeniden oluştur
results_df = pd.DataFrame(results)

results_df

In [ ]:
# ==========================================
# 116. DONANIM METRİKLERİNİ EKLE
# ==========================================

hardware_results = pd.DataFrame({
    "Model": [
        "MLP",
        "LSTM",
        "GRU",
        "BiLSTM"
    ],

    "Bias_kW": [
        mlp_bias,
        lstm_bias,
        gru_bias,
        bilstm_bias
    ],

    "FP32_KB": [
        mlp_fp32_memory_kb,
        lstm_fp32_memory_kb,
        gru_fp32_memory_kb,
        bilstm_fp32_memory_kb
    ],

    "INT16_KB": [
        mlp_int16_memory_kb,
        lstm_int16_memory_kb,
        gru_int16_memory_kb,
        bilstm_int16_memory_kb
    ],

    "INT8_KB": [
        mlp_int8_memory_kb,
        lstm_int8_memory_kb,
        gru_int8_memory_kb,
        bilstm_int8_memory_kb
    ],

    "CPU_Latency_us_Preliminary": [
        avg_time_us,
        lstm_avg_time_us,
        gru_avg_time_us,
        bilstm_avg_time_us
    ]
})

hardware_results

In [ ]:
# ==========================================
# 117. ANA MODEL KARŞILAŞTIRMA TABLOSU
# ==========================================

comparison_df = results_df.merge(
    hardware_results,
    on="Model"
)

comparison_df = comparison_df[
    [
        "Model",
        "MAE",
        "RMSE",
        "R2",
        "MAPE",
        "SMAPE",
        "Bias_kW",
        "Parameters",
        "FP32_KB",
        "INT16_KB",
        "INT8_KB",
        "CPU_Latency_us_Preliminary",
        "Best_Epoch"
    ]
]

comparison_df.round({
    "MAE": 4,
    "RMSE": 4,
    "R2": 4,
    "MAPE": 2,
    "SMAPE": 2,
    "Bias_kW": 4,
    "FP32_KB": 2,
    "INT16_KB": 2,
    "INT8_KB": 2,
    "CPU_Latency_us_Preliminary": 2
})

In [ ]:
# ==========================================
# 118. SONUÇLARI KAYDET
# ==========================================

comparison_df.to_csv(
    "model_karsilastirma_sonuclari.csv",
    index=False
)

print("Kaydedildi: model_karsilastirma_sonuclari.csv")

In [ ]:
# ==========================================
# 119. R² KARŞILAŞTIRMASI
# ==========================================

models = comparison_df["Model"]
r2_values = comparison_df["R2"]

plt.figure(figsize=(8, 5))

bars = plt.bar(
    models,
    r2_values
)

plt.ylabel("R² Skoru")
plt.xlabel("Model")
plt.title("Modellerin R² Performans Karşılaştırması")

plt.ylim(
    max(0, r2_values.min() - 0.1),
    min(1, r2_values.max() + 0.1)
)

for bar, value in zip(bars, r2_values):

    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.005,
        f"{value:.3f}",
        ha="center"
    )

plt.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()

plt.savefig(
    "model_R2_karsilastirma.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "model_R2_karsilastirma.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 120. MAE / RMSE KARŞILAŞTIRMASI
# ==========================================

x = np.arange(
    len(comparison_df)
)

width = 0.35

plt.figure(figsize=(9, 5))

plt.bar(
    x - width/2,
    comparison_df["MAE"],
    width,
    label="MAE"
)

plt.bar(
    x + width/2,
    comparison_df["RMSE"],
    width,
    label="RMSE"
)

plt.xticks(
    x,
    comparison_df["Model"]
)

plt.xlabel("Model")
plt.ylabel("Hata (kW)")

plt.title(
    "Modellerin MAE ve RMSE Karşılaştırması"
)

plt.legend()
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()

plt.savefig(
    "MAE_RMSE_karsilastirma.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "MAE_RMSE_karsilastirma.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 121. MAPE / SMAPE
# ==========================================

x = np.arange(
    len(comparison_df)
)

width = 0.35

plt.figure(figsize=(9, 5))

plt.bar(
    x - width/2,
    comparison_df["MAPE"],
    width,
    label="MAPE"
)

plt.bar(
    x + width/2,
    comparison_df["SMAPE"],
    width,
    label="SMAPE"
)

plt.xticks(
    x,
    comparison_df["Model"]
)

plt.xlabel("Model")
plt.ylabel("Hata (%)")

plt.title(
    "Modellerin MAPE ve SMAPE Karşılaştırması"
)

plt.legend()
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()

plt.savefig(
    "MAPE_SMAPE_karsilastirma.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "MAPE_SMAPE_karsilastirma.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 122. PARAMETRE - PERFORMANS
# ==========================================

plt.figure(figsize=(9, 6))

for _, row in comparison_df.iterrows():

    plt.scatter(
        row["Parameters"],
        row["R2"],
        s=120
    )

    plt.annotate(
        row["Model"],
        (
            row["Parameters"],
            row["R2"]
        ),
        xytext=(6, 5),
        textcoords="offset points"
    )

plt.xlabel("Toplam Parametre Sayısı")
plt.ylabel("R² Skoru")

plt.title(
    "Model Karmaşıklığı ve Tahmin Performansı"
)

plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    "parametre_R2_karsilastirma.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "parametre_R2_karsilastirma.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 123. INT8 BELLEK KARŞILAŞTIRMASI
# ==========================================

plt.figure(figsize=(8, 5))

bars = plt.bar(
    comparison_df["Model"],
    comparison_df["INT8_KB"]
)

plt.xlabel("Model")
plt.ylabel("Teorik Ağırlık Belleği (KB)")

plt.title(
    "INT8 Nicemleme Sonrası Teorik Model Belleği"
)

for bar, value in zip(
    bars,
    comparison_df["INT8_KB"]
):

    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.3,
        f"{value:.1f}",
        ha="center"
    )

plt.grid(axis="y", alpha=0.25)
plt.tight_layout()

plt.savefig(
    "INT8_bellek_karsilastirma.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "INT8_bellek_karsilastirma.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 125. STANDART CPU LATENCY BENCHMARK
# ==========================================

import copy
import time
import numpy as np
import torch

# Daha karşılaştırılabilir CPU ölçümü için
torch.set_num_threads(1)

def benchmark_latency(
    model,
    sample,
    warmup=200,
    runs=2000,
    repeats=7
):

    model_cpu = copy.deepcopy(model).cpu()
    model_cpu.eval()

    sample_cpu = sample.cpu()

    # Warm-up
    with torch.inference_mode():
        for _ in range(warmup):
            _ = model_cpu(sample_cpu)

    repeat_latencies = []

    for repeat in range(repeats):

        start = time.perf_counter()

        with torch.inference_mode():
            for _ in range(runs):
                _ = model_cpu(sample_cpu)

        end = time.perf_counter()

        latency_us = (
            (end - start)
            / runs
            * 1_000_000
        )

        repeat_latencies.append(latency_us)

    repeat_latencies = np.array(
        repeat_latencies
    )

    return {
        "mean_us": np.mean(repeat_latencies),
        "median_us": np.median(repeat_latencies),
        "std_us": np.std(repeat_latencies),
        "min_us": np.min(repeat_latencies),
        "max_us": np.max(repeat_latencies)
    }

In [ ]:
# ==========================================
# 126. TÜM MODELLERİ BENCHMARK ET
# ==========================================

benchmark_sample = X_test_tensor[0:1]

models_for_benchmark = {
    "MLP": mlp_model,
    "LSTM": lstm_model,
    "GRU": gru_model,
    "BiLSTM": bilstm_model
}

latency_results = {}

for model_name, model in models_for_benchmark.items():

    result = benchmark_latency(
        model,
        benchmark_sample
    )

    latency_results[model_name] = result

    print("=" * 55)
    print(model_name)
    print("=" * 55)

    print(
        f"Ortalama : "
        f"{result['mean_us']:.2f} µs"
    )

    print(
        f"Medyan   : "
        f"{result['median_us']:.2f} µs"
    )

    print(
        f"Std      : "
        f"{result['std_us']:.2f} µs"
    )

    print(
        f"Min      : "
        f"{result['min_us']:.2f} µs"
    )

    print(
        f"Max      : "
        f"{result['max_us']:.2f} µs"
    )

In [ ]:
# ==========================================
# 127. LATENCY TABLOSU
# ==========================================

latency_df = pd.DataFrame([
    {
        "Model": name,
        "Mean_us": values["mean_us"],
        "Median_us": values["median_us"],
        "Std_us": values["std_us"],
        "Min_us": values["min_us"],
        "Max_us": values["max_us"]
    }
    for name, values
    in latency_results.items()
])

latency_df.round(2)

In [ ]:
# ==========================================
# 128. ANA TABLOYA STANDART LATENCY EKLE
# ==========================================

comparison_df = comparison_df.drop(
    columns=[
        "CPU_Latency_us_Preliminary"
    ],
    errors="ignore"
)

comparison_df = comparison_df.merge(
    latency_df[
        [
            "Model",
            "Median_us"
        ]
    ],
    on="Model",
    how="left"
)

comparison_df = comparison_df.rename(
    columns={
        "Median_us":
        "CPU_Latency_us"
    }
)

comparison_df.round(4)

In [ ]:
# ==========================================
# 129. TEORİK MAC HESABI
# ==========================================

T = SEQ_LEN
I = INPUT_SIZE
H = HIDDEN_SIZE


# ---------- MLP ----------

mlp_macs = (
    (T * I) * 64
    + 64 * 32
    + 32 * 1
)


# ---------- LSTM ----------

lstm_macs_per_step = (
    4 * (
        I * H
        + H * H
    )
)

lstm_macs = (
    T * lstm_macs_per_step
    + H
)


# ---------- GRU ----------

gru_macs_per_step = (
    3 * (
        I * H
        + H * H
    )
)

gru_macs = (
    T * gru_macs_per_step
    + H
)


# ---------- BiLSTM ----------

bilstm_macs = (
    2 * T * lstm_macs_per_step
    + (2 * H)
)


mac_results = pd.DataFrame({
    "Model": [
        "MLP",
        "LSTM",
        "GRU",
        "BiLSTM"
    ],

    "Approx_MACs": [
        mlp_macs,
        lstm_macs,
        gru_macs,
        bilstm_macs
    ]
})

mac_results["Approx_MACs_k"] = (
    mac_results["Approx_MACs"] / 1000
)

mac_results

In [ ]:
# ==========================================
# 131. MAC SONUÇLARINI EKLE
# ==========================================

comparison_df = comparison_df.merge(
    mac_results[
        [
            "Model",
            "Approx_MACs"
        ]
    ],
    on="Model",
    how="left"
)

comparison_df.round(4)

In [ ]:
# ==========================================
# 132. MAC KARŞILAŞTIRMA GRAFİĞİ
# ==========================================

plt.figure(figsize=(8, 5))

bars = plt.bar(
    mac_results["Model"],
    mac_results["Approx_MACs_k"]
)

plt.xlabel("Model")

plt.ylabel(
    "Yaklaşık MAC Sayısı (bin)"
)

plt.title(
    "Modellerin Teorik Hesaplama Karmaşıklığı"
)

for bar, value in zip(
    bars,
    mac_results["Approx_MACs_k"]
):

    plt.text(
        bar.get_x()
        + bar.get_width()/2,

        bar.get_height() + 10,

        f"{value:.1f}K",

        ha="center"
    )

plt.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()

plt.savefig(
    "MAC_karsilastirma.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "MAC_karsilastirma.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 132. MAC KARŞILAŞTIRMA GRAFİĞİ
# ==========================================

plt.figure(figsize=(8, 5))

bars = plt.bar(
    mac_results["Model"],
    mac_results["Approx_MACs_k"]
)

plt.xlabel("Model")

plt.ylabel(
    "Yaklaşık MAC Sayısı (bin)"
)

plt.title(
    "Modellerin Teorik Hesaplama Karmaşıklığı"
)

for bar, value in zip(
    bars,
    mac_results["Approx_MACs_k"]
):

    plt.text(
        bar.get_x()
        + bar.get_width()/2,

        bar.get_height() + 10,

        f"{value:.1f}K",

        ha="center"
    )

plt.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()

plt.savefig(
    "MAC_karsilastirma.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "MAC_karsilastirma.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 134. GÜNCEL SONUÇ TABLOSU
# ==========================================

comparison_df.to_csv(
    "nihai_model_karsilastirma.csv",
    index=False
)

print(
    "Kaydedildi: nihai_model_karsilastirma.csv"
)

In [ ]:
# ==========================================
# /content ALTINDAKİ TÜM GRAFİKLERİ ZIP YAP
# ==========================================

import os
import glob
import shutil
from google.colab import files

source_dir = "/content"
output_dir = "/content/makale_grafikleri"

# Eski klasör varsa sil
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir)

# PNG ve PDF dosyalarını topla
for extension in ["*.png", "*.pdf"]:

    for file_path in glob.glob(
        os.path.join(source_dir, extension)
    ):

        shutil.copy2(
            file_path,
            output_dir
        )

# Kaç dosya bulunduğunu göster
collected_files = os.listdir(output_dir)

print("Toplanan dosya sayısı:", len(collected_files))

for file_name in sorted(collected_files):
    print(file_name)

# ZIP oluştur
zip_path = shutil.make_archive(
    "/content/makale_grafikleri",
    "zip",
    output_dir
)

print("\nZIP hazır:", zip_path)

# Bilgisayara indir
files.download(zip_path)

In [ ]:
# ==========================================
# 135. ACF / PACF KÜTÜPHANELERİ
# ==========================================

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf, pacf

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# ==========================================
# 136. ACF / PACF ANALİZ SERİSİ
# ==========================================

acf_series = (
    train_df["Global_active_power"]
    .dropna()
    .astype(float)
)

print("ACF/PACF örnek sayısı:", len(acf_series))
print("Başlangıç:", acf_series.index.min())
print("Bitiş:", acf_series.index.max())

In [ ]:
# ==========================================
# 137. ACF - İLK 48 SAAT
# ==========================================

fig, ax = plt.subplots(figsize=(12, 5))

plot_acf(
    acf_series,
    lags=48,
    ax=ax,
    zero=False
)

ax.set_title(
    "Saatlik Aktif Güç İçin Otokorelasyon (ACF) - İlk 48 Saat"
)

ax.set_xlabel("Gecikme (Saat)")
ax.set_ylabel("Otokorelasyon")

plt.grid(alpha=0.2)
plt.tight_layout()

plt.savefig(
    "ACF_48_saat.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "ACF_48_saat.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 138. ACF - 168 SAAT / 1 HAFTA
# ==========================================

fig, ax = plt.subplots(figsize=(14, 5))

plot_acf(
    acf_series,
    lags=168,
    ax=ax,
    zero=False
)

ax.set_title(
    "Saatlik Aktif Güç İçin Otokorelasyon (ACF) - 168 Saat"
)

ax.set_xlabel("Gecikme (Saat)")
ax.set_ylabel("Otokorelasyon")

# Günlük sınırları göster
for lag in [24, 48, 72, 96, 120, 144, 168]:
    ax.axvline(
        lag,
        linestyle="--",
        alpha=0.25
    )

plt.tight_layout()

plt.savefig(
    "ACF_168_saat.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "ACF_168_saat.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 139. KRİTİK ACF DEĞERLERİ
# ==========================================

acf_values = acf(
    acf_series,
    nlags=168,
    fft=True
)

important_lags = [
    1,
    2,
    3,
    6,
    12,
    24,
    48,
    72,
    168
]

print("=" * 45)
print("KRİTİK ACF DEĞERLERİ")
print("=" * 45)

for lag in important_lags:
    print(
        f"Lag {lag:3d} saat : "
        f"{acf_values[lag]:.4f}"
    )

In [ ]:
# ==========================================
# 140. PACF - 48 SAAT
# ==========================================

fig, ax = plt.subplots(figsize=(12, 5))

plot_pacf(
    acf_series,
    lags=48,
    method="ywm",
    ax=ax,
    zero=False
)

ax.set_title(
    "Saatlik Aktif Güç İçin Kısmi Otokorelasyon (PACF)"
)

ax.set_xlabel("Gecikme (Saat)")
ax.set_ylabel("Kısmi Otokorelasyon")

plt.grid(alpha=0.2)
plt.tight_layout()

plt.savefig(
    "PACF_48_saat.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "PACF_48_saat.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 141. KRİTİK PACF DEĞERLERİ
# ==========================================

pacf_values = pacf(
    acf_series,
    nlags=168,
    method="ywm"
)

print("=" * 45)
print("KRİTİK PACF DEĞERLERİ")
print("=" * 45)

for lag in important_lags:
    print(
        f"Lag {lag:3d} saat : "
        f"{pacf_values[lag]:.4f}"
    )

In [ ]:
# ==========================================
# 142. ACF / PACF ÖZET TABLOSU
# ==========================================

acf_pacf_table = pd.DataFrame({
    "Lag_saat": important_lags,
    "ACF": [
        acf_values[lag]
        for lag in important_lags
    ],
    "PACF": [
        pacf_values[lag]
        for lag in important_lags
    ]
})

acf_pacf_table["ACF"] = (
    acf_pacf_table["ACF"].round(4)
)

acf_pacf_table["PACF"] = (
    acf_pacf_table["PACF"].round(4)
)

acf_pacf_table

In [ ]:
acf_pacf_table.to_csv(
    "acf_pacf_kritik_laglar.csv",
    index=False
)

In [ ]:
# ==========================================
# 145. DENEY TEKRARLANABİLİRLİĞİ
# ==========================================

import random
import numpy as np
import torch

def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
# ==========================================
# 146. VALIDATION TAHMİN FONKSİYONU
# ==========================================

def evaluate_loader(model, loader):

    model.eval()

    predictions = []
    actuals = []

    with torch.inference_mode():

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(device)

            pred = model(X_batch)

            predictions.append(
                pred.cpu().numpy()
            )

            actuals.append(
                y_batch.numpy()
            )

    predictions = np.concatenate(
        predictions,
        axis=0
    )

    actuals = np.concatenate(
        actuals,
        axis=0
    )

    return predictions, actuals

In [ ]:
# ==========================================
# 147. SEQUENCE LENGTH ADAYLARI
# ==========================================

SEQ_CANDIDATES = [
    6,
    12,
    24,
    48,
    168
]

print(SEQ_CANDIDATES)

In [ ]:
# ==========================================
# 148. SEQUENCE LENGTH ABLATION - LSTM
# ==========================================

sequence_results = []

for seq_len in SEQ_CANDIDATES:

    print("\n" + "=" * 65)
    print(f"SEQUENCE LENGTH = {seq_len}")
    print("=" * 65)

    # Aynı başlangıç seed'i
    set_seed(42)

    # --------------------------
    # TRAIN SEQUENCE
    # --------------------------

    X_train_seq, y_train_seq = create_sequences(
        X_train_scaled,
        y_train_scaled,
        seq_len
    )

    # --------------------------
    # VALIDATION SEQUENCE
    # --------------------------

    X_val_seq, y_val_seq = create_eval_sequences(
        X_train_scaled,
        X_val_scaled,
        y_val_scaled,
        seq_len
    )

    # Tensor
    X_train_t = torch.tensor(
        X_train_seq,
        dtype=torch.float32
    )

    y_train_t = torch.tensor(
        y_train_seq,
        dtype=torch.float32
    )

    X_val_t = torch.tensor(
        X_val_seq,
        dtype=torch.float32
    )

    y_val_t = torch.tensor(
        y_val_seq,
        dtype=torch.float32
    )

    # Dataset
    train_ds = TensorDataset(
        X_train_t,
        y_train_t
    )

    val_ds = TensorDataset(
        X_val_t,
        y_val_t
    )

    # Loader
    train_dl = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    val_dl = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    # --------------------------
    # YENİ LSTM
    # --------------------------

    model = LSTMModel(
        input_size=INPUT_SIZE,
        hidden_size=HIDDEN_SIZE
    ).to(device)

    # --------------------------
    # EĞİTİM
    # --------------------------

    model, train_losses, val_losses = train_model(
        model,
        train_dl,
        val_dl,
        learning_rate=LEARNING_RATE,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE
    )

    best_epoch_seq = (
        np.argmin(val_losses) + 1
    )

    # --------------------------
    # VALIDATION TAHMİNİ
    # --------------------------

    pred_scaled, true_scaled = evaluate_loader(
        model,
        val_dl
    )

    pred = target_scaler.inverse_transform(
        pred_scaled
    ).flatten()

    true = target_scaler.inverse_transform(
        true_scaled
    ).flatten()

    # --------------------------
    # METRİKLER
    # --------------------------

    (
        mae,
        rmse,
        r2,
        mape,
        smape
    ) = calculate_metrics(
        true,
        pred
    )

    sequence_results.append({
        "Sequence_Length": seq_len,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAPE": mape,
        "SMAPE": smape,
        "Best_Epoch": best_epoch_seq
    })

    print("\nVALIDATION SONUÇLARI")
    print(f"MAE   : {mae:.6f}")
    print(f"RMSE  : {rmse:.6f}")
    print(f"R²    : {r2:.6f}")
    print(f"SMAPE : {smape:.3f}%")

In [ ]:
# ==========================================
# 149. SEQUENCE LENGTH SONUÇ TABLOSU
# ==========================================

sequence_results_df = pd.DataFrame(
    sequence_results
)

sequence_results_df = sequence_results_df.sort_values(
    "Sequence_Length"
).reset_index(drop=True)

sequence_results_df.round({
    "MAE": 4,
    "RMSE": 4,
    "R2": 4,
    "MAPE": 2,
    "SMAPE": 2
})

In [ ]:
# ==========================================
# 151. SEQUENCE LENGTH - RMSE
# ==========================================

plt.figure(figsize=(9, 5))

plt.plot(
    sequence_results_df["Sequence_Length"],
    sequence_results_df["RMSE"],
    marker="o",
    linewidth=2
)

plt.xlabel(
    "Giriş Penceresi Uzunluğu (Saat)"
)

plt.ylabel(
    "Validation RMSE (kW)"
)

plt.title(
    "Giriş Penceresi Uzunluğunun LSTM Performansına Etkisi"
)

plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    "sequence_length_RMSE.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "sequence_length_RMSE.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 152. SEQUENCE LENGTH - R²
# ==========================================

plt.figure(figsize=(9, 5))

plt.plot(
    sequence_results_df["Sequence_Length"],
    sequence_results_df["R2"],
    marker="o",
    linewidth=2
)

plt.xlabel(
    "Giriş Penceresi Uzunluğu (Saat)"
)

plt.ylabel(
    "Validation R²"
)

plt.title(
    "Giriş Penceresi Uzunluğunun R² Üzerindeki Etkisi"
)

plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    "sequence_length_R2.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "sequence_length_R2.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
sequence_results_df.to_csv(
    "sequence_length_ablation.csv",
    index=False
)

print(
    "Kaydedildi: sequence_length_ablation.csv"
)

In [ ]:
# ==========================================
# 155. FEATURE ABLATION GRUPLARI
# ==========================================

FEATURE_SETS = {

    "Full_12": [
        "Global_active_power",
        "Global_reactive_power",
        "Voltage",
        "Global_intensity",
        "Sub_metering_1",
        "Sub_metering_2",
        "Sub_metering_3",
        "hour_sin",
        "hour_cos",
        "dow_sin",
        "dow_cos",
        "is_weekend"
    ],

    "No_Intensity_11": [
        "Global_active_power",
        "Global_reactive_power",
        "Voltage",
        "Sub_metering_1",
        "Sub_metering_2",
        "Sub_metering_3",
        "hour_sin",
        "hour_cos",
        "dow_sin",
        "dow_cos",
        "is_weekend"
    ],

    "Electrical_7": [
        "Global_active_power",
        "Global_reactive_power",
        "Voltage",
        "Global_intensity",
        "Sub_metering_1",
        "Sub_metering_2",
        "Sub_metering_3"
    ],

    "Power_Time_6": [
        "Global_active_power",
        "hour_sin",
        "hour_cos",
        "dow_sin",
        "dow_cos",
        "is_weekend"
    ],

    "Power_Only_1": [
        "Global_active_power"
    ]
}

for name, features in FEATURE_SETS.items():
    print(
        f"{name:20s} → "
        f"{len(features)} özellik"
    )

In [ ]:
# ==========================================
# 156. FEATURE INDEX
# ==========================================

feature_to_index = {
    feature: i
    for i, feature in enumerate(FEATURES)
}

print(feature_to_index)

In [ ]:
# ==========================================
# 157. FEATURE ABLATION - LSTM
# ==========================================

FEATURE_SEQ_LEN = 24

feature_ablation_results = []

for set_name, selected_features in FEATURE_SETS.items():

    print("\n" + "=" * 70)
    print(set_name)
    print("=" * 70)

    set_seed(42)

    # ---------------------------------
    # İLGİLİ SÜTUNLARI BUL
    # ---------------------------------

    indices = [
        feature_to_index[f]
        for f in selected_features
    ]

    X_train_selected = X_train_scaled[:, indices]
    X_val_selected = X_val_scaled[:, indices]

    # ---------------------------------
    # SEQUENCE
    # ---------------------------------

    X_train_seq, y_train_seq = create_sequences(
        X_train_selected,
        y_train_scaled,
        FEATURE_SEQ_LEN
    )

    X_val_seq, y_val_seq = create_eval_sequences(
        X_train_selected,
        X_val_selected,
        y_val_scaled,
        FEATURE_SEQ_LEN
    )

    # ---------------------------------
    # TENSOR
    # ---------------------------------

    X_train_t = torch.tensor(
        X_train_seq,
        dtype=torch.float32
    )

    y_train_t = torch.tensor(
        y_train_seq,
        dtype=torch.float32
    )

    X_val_t = torch.tensor(
        X_val_seq,
        dtype=torch.float32
    )

    y_val_t = torch.tensor(
        y_val_seq,
        dtype=torch.float32
    )

    # ---------------------------------
    # DATALOADER
    # ---------------------------------

    train_dl = DataLoader(
        TensorDataset(
            X_train_t,
            y_train_t
        ),
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    val_dl = DataLoader(
        TensorDataset(
            X_val_t,
            y_val_t
        ),
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    # ---------------------------------
    # MODEL
    # ---------------------------------

    model = LSTMModel(
        input_size=len(selected_features),
        hidden_size=HIDDEN_SIZE
    ).to(device)

    # ---------------------------------
    # EĞİTİM
    # ---------------------------------

    model, train_losses, val_losses = train_model(
        model,
        train_dl,
        val_dl,
        learning_rate=LEARNING_RATE,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE
    )

    best_epoch_ablation = (
        np.argmin(val_losses) + 1
    )

    # ---------------------------------
    # VALIDATION TAHMİNİ
    # ---------------------------------

    pred_scaled, true_scaled = evaluate_loader(
        model,
        val_dl
    )

    pred = target_scaler.inverse_transform(
        pred_scaled
    ).flatten()

    true = target_scaler.inverse_transform(
        true_scaled
    ).flatten()

    (
        mae,
        rmse,
        r2,
        mape,
        smape
    ) = calculate_metrics(
        true,
        pred
    )

    # ---------------------------------
    # PARAMETRE VE MAC
    # ---------------------------------

    parameters = count_parameters(model)

    I_ablation = len(selected_features)

    approx_macs = (
        FEATURE_SEQ_LEN
        * 4
        * (
            I_ablation * HIDDEN_SIZE
            + HIDDEN_SIZE * HIDDEN_SIZE
        )
        + HIDDEN_SIZE
    )

    feature_ablation_results.append({

        "Feature_Set": set_name,
        "Feature_Count": len(selected_features),

        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAPE": mape,
        "SMAPE": smape,

        "Parameters": parameters,
        "Approx_MACs": approx_macs,

        "Best_Epoch": best_epoch_ablation
    })

    print("\nVALIDATION")
    print(f"Feature sayısı : {len(selected_features)}")
    print(f"MAE            : {mae:.6f}")
    print(f"RMSE           : {rmse:.6f}")
    print(f"R²             : {r2:.6f}")
    print(f"SMAPE          : {smape:.3f}%")
    print(f"Parametre      : {parameters:,}")
    print(f"Yaklaşık MAC   : {approx_macs:,}")

In [ ]:
# ==========================================
# 158. FEATURE ABLATION TABLOSU
# ==========================================

feature_ablation_df = pd.DataFrame(
    feature_ablation_results
)

feature_ablation_df.round({
    "MAE": 4,
    "RMSE": 4,
    "R2": 4,
    "MAPE": 2,
    "SMAPE": 2
})

In [ ]:
# ==========================================
# 159. EN İYİ FEATURE SET
# ==========================================

best_feature_row = feature_ablation_df.loc[
    feature_ablation_df["RMSE"].idxmin()
]

BEST_FEATURE_SET = (
    best_feature_row["Feature_Set"]
)

print("=" * 60)
print("EN İYİ FEATURE SET")
print("=" * 60)

print(
    "Feature set:",
    BEST_FEATURE_SET
)

print(
    "Feature sayısı:",
    int(best_feature_row["Feature_Count"])
)

print(
    f"RMSE:"
    f" {best_feature_row['RMSE']:.6f}"
)

print(
    f"R²:"
    f" {best_feature_row['R2']:.6f}"
)

print(
    f"SMAPE:"
    f" {best_feature_row['SMAPE']:.3f}%"
)

In [ ]:
# ==========================================
# 160. FEATURE COUNT - R²
# ==========================================

plt.figure(figsize=(9, 6))

for _, row in feature_ablation_df.iterrows():

    plt.scatter(
        row["Feature_Count"],
        row["R2"],
        s=120
    )

    plt.annotate(
        row["Feature_Set"],
        (
            row["Feature_Count"],
            row["R2"]
        ),
        xytext=(6, 5),
        textcoords="offset points"
    )

plt.xlabel("Özellik Sayısı")
plt.ylabel("Validation R²")

plt.title(
    "Özellik Sayısı ve Tahmin Performansı"
)

plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    "feature_ablation_R2.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "feature_ablation_R2.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 161. FEATURE ABLATION RMSE
# ==========================================

plt.figure(figsize=(10, 5))

bars = plt.bar(
    feature_ablation_df["Feature_Set"],
    feature_ablation_df["RMSE"]
)

plt.ylabel("Validation RMSE (kW)")
plt.xlabel("Özellik Kümesi")

plt.title(
    "Özellik Seçiminin LSTM Tahmin Performansına Etkisi"
)

plt.xticks(rotation=15)

for bar, value in zip(
    bars,
    feature_ablation_df["RMSE"]
):

    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.002,
        f"{value:.3f}",
        ha="center"
    )

plt.grid(axis="y", alpha=0.25)
plt.tight_layout()

plt.savefig(
    "feature_ablation_RMSE.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "feature_ablation_RMSE.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
feature_ablation_df.to_csv(
    "feature_ablation_results.csv",
    index=False
)

print(
    "Kaydedildi: feature_ablation_results.csv"
)

In [ ]:
# ==========================================
# 163. POWER_TIME_6 YAKINSAMA KONTROLÜ
# ==========================================

set_seed(42)

selected_features = FEATURE_SETS["Power_Time_6"]

indices = [
    feature_to_index[f]
    for f in selected_features
]

X_train_selected = X_train_scaled[:, indices]
X_val_selected = X_val_scaled[:, indices]

# Sequence
X_train_seq, y_train_seq = create_sequences(
    X_train_selected,
    y_train_scaled,
    24
)

X_val_seq, y_val_seq = create_eval_sequences(
    X_train_selected,
    X_val_selected,
    y_val_scaled,
    24
)

# Tensor
X_train_t = torch.tensor(
    X_train_seq,
    dtype=torch.float32
)

y_train_t = torch.tensor(
    y_train_seq,
    dtype=torch.float32
)

X_val_t = torch.tensor(
    X_val_seq,
    dtype=torch.float32
)

y_val_t = torch.tensor(
    y_val_seq,
    dtype=torch.float32
)

# DataLoader
train_dl = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=BATCH_SIZE,
    shuffle=False
)

val_dl = DataLoader(
    TensorDataset(X_val_t, y_val_t),
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Model
power_time_model = LSTMModel(
    input_size=len(selected_features),
    hidden_size=HIDDEN_SIZE
).to(device)

# Bu kez 200 epoch üst sınırı
power_time_model, pt_train_losses, pt_val_losses = train_model(
    power_time_model,
    train_dl,
    val_dl,
    learning_rate=LEARNING_RATE,
    max_epochs=200,
    patience=PATIENCE
)

In [ ]:
# ==========================================
# 164. POWER_TIME_6 YENİ SONUÇ
# ==========================================

pt_best_epoch = np.argmin(
    pt_val_losses
) + 1

pred_scaled, true_scaled = evaluate_loader(
    power_time_model,
    val_dl
)

pred = target_scaler.inverse_transform(
    pred_scaled
).flatten()

true = target_scaler.inverse_transform(
    true_scaled
).flatten()

(
    pt_mae,
    pt_rmse,
    pt_r2,
    pt_mape,
    pt_smape
) = calculate_metrics(
    true,
    pred
)

print("=" * 60)
print("POWER_TIME_6 - UZATILMIŞ EĞİTİM")
print("=" * 60)

print(f"En iyi epoch : {pt_best_epoch}")
print(f"MAE          : {pt_mae:.6f}")
print(f"RMSE         : {pt_rmse:.6f}")
print(f"R²           : {pt_r2:.6f}")
print(f"MAPE         : {pt_mape:.3f}%")
print(f"SMAPE        : {pt_smape:.3f}%")

In [ ]:
# ==========================================
# 165. POWER_TIME_6 SATIRINI GÜNCELLE
# ==========================================

mask = (
    feature_ablation_df["Feature_Set"]
    == "Power_Time_6"
)

feature_ablation_df.loc[
    mask,
    [
        "MAE",
        "RMSE",
        "R2",
        "MAPE",
        "SMAPE",
        "Best_Epoch"
    ]
] = [
    pt_mae,
    pt_rmse,
    pt_r2,
    pt_mape,
    pt_smape,
    pt_best_epoch
]

feature_ablation_df.round({
    "MAE": 4,
    "RMSE": 4,
    "R2": 4,
    "MAPE": 2,
    "SMAPE": 2
})

In [ ]:
best_feature_row = feature_ablation_df.loc[
    feature_ablation_df["RMSE"].idxmin()
]

print(
    "En iyi özellik seti:",
    best_feature_row["Feature_Set"]
)

print(
    "Validation RMSE:",
    best_feature_row["RMSE"]
)

print(
    "Validation R²:",
    best_feature_row["R2"]
)

In [ ]:
# ==========================================
# 166. FINAL RANDOM SEEDS
# ==========================================

FINAL_SEEDS = [
    11,
    22,
    33,
    44,
    55
]

print("Kullanılacak seed'ler:", FINAL_SEEDS)

In [ ]:
# ==========================================
# 167. MODEL FACTORY
# ==========================================

def create_model(model_name):

    if model_name == "MLP":

        model = MLPModel(
            seq_len=24,
            input_size=12
        )

    elif model_name == "LSTM":

        model = LSTMModel(
            input_size=12,
            hidden_size=64
        )

    elif model_name == "GRU":

        model = GRUModel(
            input_size=12,
            hidden_size=64
        )

    elif model_name == "BiLSTM":

        # Düzeltilmiş BiLSTM kullanılıyor
        model = BiLSTMModelCorrected(
            input_size=12,
            hidden_size=64
        )

    else:
        raise ValueError(
            f"Bilinmeyen model: {model_name}"
        )

    return model.to(device)

In [ ]:
# ==========================================
# 168. TEK SEED DENEYİ
# ==========================================

def run_single_experiment(
    model_name,
    seed
):

    set_seed(seed)

    # -------------------------------
    # MODEL
    # -------------------------------

    model = create_model(
        model_name
    )

    # -------------------------------
    # EĞİTİM
    # -------------------------------

    model, train_losses, val_losses = train_model(
        model,
        train_loader,
        val_loader,
        learning_rate=0.001,
        max_epochs=150,
        patience=10
    )

    best_epoch = (
        np.argmin(val_losses) + 1
    )

    # ===============================
    # VALIDATION
    # ===============================

    val_pred_scaled, val_true_scaled = predict_model(
        model,
        val_loader
    )

    val_pred = target_scaler.inverse_transform(
        val_pred_scaled
    ).flatten()

    val_true = target_scaler.inverse_transform(
        val_true_scaled
    ).flatten()

    (
        val_mae,
        val_rmse,
        val_r2,
        val_mape,
        val_smape
    ) = calculate_metrics(
        val_true,
        val_pred
    )

    # ===============================
    # TEST
    # ===============================

    test_pred_scaled, test_true_scaled = predict_model(
        model,
        test_loader
    )

    test_pred = target_scaler.inverse_transform(
        test_pred_scaled
    ).flatten()

    test_true = target_scaler.inverse_transform(
        test_true_scaled
    ).flatten()

    (
        test_mae,
        test_rmse,
        test_r2,
        test_mape,
        test_smape
    ) = calculate_metrics(
        test_true,
        test_pred
    )

    # -------------------------------
    # BIAS
    # -------------------------------

    test_bias = np.mean(
        test_pred - test_true
    )

    # -------------------------------
    # MODELİ KAYDET
    # -------------------------------

    model_path = (
        f"{model_name}_seed_{seed}.pth"
    )

    torch.save(
        model.state_dict(),
        model_path
    )

    result = {

        "Model": model_name,
        "Seed": seed,

        "Best_Epoch": best_epoch,

        "Val_MAE": val_mae,
        "Val_RMSE": val_rmse,
        "Val_R2": val_r2,
        "Val_MAPE": val_mape,
        "Val_SMAPE": val_smape,

        "Test_MAE": test_mae,
        "Test_RMSE": test_rmse,
        "Test_R2": test_r2,
        "Test_MAPE": test_mape,
        "Test_SMAPE": test_smape,
        "Test_Bias": test_bias,

        "Parameters":
        count_parameters(model)
    }

    return result

In [ ]:
# ==========================================
# 169. FINAL MULTI-SEED EXPERIMENT
# ==========================================

FINAL_MODELS = [
    "MLP",
    "LSTM",
    "GRU",
    "BiLSTM"
]

multi_seed_results = []

for model_name in FINAL_MODELS:

    print("\n")
    print("#" * 75)
    print(f"MODEL: {model_name}")
    print("#" * 75)

    for seed in FINAL_SEEDS:

        print("\n" + "=" * 65)
        print(
            f"{model_name} | SEED = {seed}"
        )
        print("=" * 65)

        result = run_single_experiment(
            model_name,
            seed
        )

        multi_seed_results.append(
            result
        )

        print("\nTEST ÖZET")

        print(
            f"RMSE : "
            f"{result['Test_RMSE']:.6f}"
        )

        print(
            f"R²   : "
            f"{result['Test_R2']:.6f}"
        )

        print(
            f"SMAPE: "
            f"{result['Test_SMAPE']:.3f}%"
        )

In [ ]:
# ==========================================
# 170. TÜM SEED SONUÇLARI
# ==========================================

multi_seed_df = pd.DataFrame(
    multi_seed_results
)

multi_seed_df.round(4)

In [ ]:
# ==========================================
# 171. MEAN ± STD
# ==========================================

metrics_to_summarize = [
    "Test_MAE",
    "Test_RMSE",
    "Test_R2",
    "Test_MAPE",
    "Test_SMAPE",
    "Test_Bias"
]

summary_rows = []

for model_name in FINAL_MODELS:

    subset = multi_seed_df[
        multi_seed_df["Model"]
        == model_name
    ]

    row = {
        "Model": model_name,
        "Parameters":
            int(subset["Parameters"].iloc[0])
    }

    for metric in metrics_to_summarize:

        row[f"{metric}_Mean"] = (
            subset[metric].mean()
        )

        row[f"{metric}_Std"] = (
            subset[metric].std(ddof=1)
        )

    summary_rows.append(row)


multi_seed_summary = pd.DataFrame(
    summary_rows
)

multi_seed_summary.round(4)

In [ ]:
# ==========================================
# 172. MAKALE FORMATINDA SONUÇ TABLOSU
# ==========================================

publication_table = pd.DataFrame({

    "Model":
        multi_seed_summary["Model"],

    "MAE":
        multi_seed_summary.apply(
            lambda r:
            f"{r['Test_MAE_Mean']:.4f} ± "
            f"{r['Test_MAE_Std']:.4f}",
            axis=1
        ),

    "RMSE":
        multi_seed_summary.apply(
            lambda r:
            f"{r['Test_RMSE_Mean']:.4f} ± "
            f"{r['Test_RMSE_Std']:.4f}",
            axis=1
        ),

    "R²":
        multi_seed_summary.apply(
            lambda r:
            f"{r['Test_R2_Mean']:.4f} ± "
            f"{r['Test_R2_Std']:.4f}",
            axis=1
        ),

    "MAPE (%)":
        multi_seed_summary.apply(
            lambda r:
            f"{r['Test_MAPE_Mean']:.2f} ± "
            f"{r['Test_MAPE_Std']:.2f}",
            axis=1
        ),

    "SMAPE (%)":
        multi_seed_summary.apply(
            lambda r:
            f"{r['Test_SMAPE_Mean']:.2f} ± "
            f"{r['Test_SMAPE_Std']:.2f}",
            axis=1
        ),

    "Parametre":
        multi_seed_summary[
            "Parameters"
        ].astype(int)
})

publication_table

In [ ]:
# ==========================================
# 173. MULTI-SEED SONUÇLARI KAYDET
# ==========================================

multi_seed_df.to_csv(
    "multi_seed_tum_sonuclar.csv",
    index=False
)

multi_seed_summary.to_csv(
    "multi_seed_ozet.csv",
    index=False
)

publication_table.to_csv(
    "makale_final_model_sonuclari.csv",
    index=False
)

print("3 sonuç dosyası kaydedildi.")

In [ ]:
# ==========================================
# 174. VALIDATION'A GÖRE DEPLOYMENT MODELİ
# ==========================================

deployment_rows = []

for model_name in FINAL_MODELS:

    subset = multi_seed_df[
        multi_seed_df["Model"] == model_name
    ]

    best_row = subset.loc[
        subset["Val_RMSE"].idxmin()
    ]

    deployment_rows.append({
        "Model": model_name,
        "Selected_Seed": int(best_row["Seed"]),
        "Val_RMSE": best_row["Val_RMSE"],
        "Val_R2": best_row["Val_R2"],
        "Test_RMSE": best_row["Test_RMSE"],
        "Test_R2": best_row["Test_R2"]
    })


deployment_df = pd.DataFrame(
    deployment_rows
)

deployment_df.round(4)

In [ ]:
# ==========================================
# 175. DEPLOYMENT CHECKPOINT YOLLARI
# ==========================================

deployment_paths = {}

for _, row in deployment_df.iterrows():

    model_name = row["Model"]
    seed = int(row["Selected_Seed"])

    path = f"{model_name}_seed_{seed}.pth"

    deployment_paths[model_name] = path

    print(
        f"{model_name:8s} → {path}"
    )

In [ ]:
# ==========================================
# 176. CHECKPOINTLERİ YÜKLE
# ==========================================

deployment_models = {}

for model_name in FINAL_MODELS:

    seed = int(
        deployment_df.loc[
            deployment_df["Model"] == model_name,
            "Selected_Seed"
        ].iloc[0]
    )

    model = create_model(model_name)

    state_dict = torch.load(
        f"{model_name}_seed_{seed}.pth",
        map_location="cpu"
    )

    model.load_state_dict(
        state_dict
    )

    model = model.cpu()
    model.eval()

    deployment_models[
        model_name
    ] = model

    print(
        f"{model_name} yüklendi | "
        f"seed={seed}"
    )

In [ ]:
# ==========================================
# 177. INT8 DYNAMIC QUANTIZATION
# ==========================================

import torch.ao.quantization as tq

quantized_models = {}

for model_name, model in deployment_models.items():

    q_model = tq.quantize_dynamic(
        model,
        {
            nn.Linear,
            nn.LSTM,
            nn.GRU
        },
        dtype=torch.qint8
    )

    quantized_models[
        model_name
    ] = q_model

    print(
        f"{model_name} INT8 quantize edildi."
    )

In [ ]:
# ==========================================
# 178. CPU QUANTIZED PREDICTION
# ==========================================

def predict_cpu_model(
    model,
    data_loader
):

    model.eval()

    predictions = []
    actuals = []

    with torch.inference_mode():

        for X_batch, y_batch in data_loader:

            X_batch = X_batch.cpu()

            output = model(
                X_batch
            )

            predictions.append(
                output.detach().cpu().numpy()
            )

            actuals.append(
                y_batch.cpu().numpy()
            )

    predictions = np.concatenate(
        predictions,
        axis=0
    )

    actuals = np.concatenate(
        actuals,
        axis=0
    )

    return predictions, actuals

In [ ]:
# ==========================================
# 179. FP32 vs INT8 DOĞRULUK
# ==========================================

quantization_results = []

for model_name in FINAL_MODELS:

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)

    fp32_model = deployment_models[
        model_name
    ]

    int8_model = quantized_models[
        model_name
    ]

    # ---------------- FP32 ----------------

    fp32_pred_scaled, true_scaled = predict_cpu_model(
        fp32_model,
        test_loader
    )

    fp32_pred = target_scaler.inverse_transform(
        fp32_pred_scaled
    ).flatten()

    true = target_scaler.inverse_transform(
        true_scaled
    ).flatten()

    (
        fp32_mae,
        fp32_rmse,
        fp32_r2,
        fp32_mape,
        fp32_smape
    ) = calculate_metrics(
        true,
        fp32_pred
    )

    # ---------------- INT8 ----------------

    int8_pred_scaled, _ = predict_cpu_model(
        int8_model,
        test_loader
    )

    int8_pred = target_scaler.inverse_transform(
        int8_pred_scaled
    ).flatten()

    (
        int8_mae,
        int8_rmse,
        int8_r2,
        int8_mape,
        int8_smape
    ) = calculate_metrics(
        true,
        int8_pred
    )

    quantization_results.append({

        "Model": model_name,

        "FP32_RMSE": fp32_rmse,
        "INT8_RMSE": int8_rmse,

        "FP32_R2": fp32_r2,
        "INT8_R2": int8_r2,

        "FP32_SMAPE": fp32_smape,
        "INT8_SMAPE": int8_smape
    })

    print(
        f"FP32 RMSE : {fp32_rmse:.6f}"
    )

    print(
        f"INT8 RMSE : {int8_rmse:.6f}"
    )

    print(
        f"FP32 R²   : {fp32_r2:.6f}"
    )

    print(
        f"INT8 R²   : {int8_r2:.6f}"
    )

In [ ]:
# ==========================================
# 179. FP32 vs INT8 DOĞRULUK
# ==========================================

quantization_results = []

for model_name in FINAL_MODELS:

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)

    fp32_model = deployment_models[
        model_name
    ]

    int8_model = quantized_models[
        model_name
    ]

    # ---------------- FP32 ----------------

    fp32_pred_scaled, true_scaled = predict_cpu_model(
        fp32_model,
        test_loader
    )

    fp32_pred = target_scaler.inverse_transform(
        fp32_pred_scaled
    ).flatten()

    true = target_scaler.inverse_transform(
        true_scaled
    ).flatten()

    (
        fp32_mae,
        fp32_rmse,
        fp32_r2,
        fp32_mape,
        fp32_smape
    ) = calculate_metrics(
        true,
        fp32_pred
    )

    # ---------------- INT8 ----------------

    int8_pred_scaled, _ = predict_cpu_model(
        int8_model,
        test_loader
    )

    int8_pred = target_scaler.inverse_transform(
        int8_pred_scaled
    ).flatten()

    (
        int8_mae,
        int8_rmse,
        int8_r2,
        int8_mape,
        int8_smape
    ) = calculate_metrics(
        true,
        int8_pred
    )

    quantization_results.append({

        "Model": model_name,

        "FP32_RMSE": fp32_rmse,
        "INT8_RMSE": int8_rmse,

        "FP32_R2": fp32_r2,
        "INT8_R2": int8_r2,

        "FP32_SMAPE": fp32_smape,
        "INT8_SMAPE": int8_smape
    })

    print(
        f"FP32 RMSE : {fp32_rmse:.6f}"
    )

    print(
        f"INT8 RMSE : {int8_rmse:.6f}"
    )

    print(
        f"FP32 R²   : {fp32_r2:.6f}"
    )

    print(
        f"INT8 R²   : {int8_r2:.6f}"
    )

In [ ]:
# ==========================================
# 181. GERÇEK DOSYA BOYUTLARI
# ==========================================

size_results = []

for model_name in FINAL_MODELS:

    fp32_path = (
        f"{model_name}_deployment_FP32.pth"
    )

    int8_path = (
        f"{model_name}_deployment_INT8.pth"
    )

    torch.save(
        deployment_models[
            model_name
        ].state_dict(),
        fp32_path
    )

    torch.save(
        quantized_models[
            model_name
        ].state_dict(),
        int8_path
    )

    fp32_size = (
        os.path.getsize(fp32_path)
        / 1024
    )

    int8_size = (
        os.path.getsize(int8_path)
        / 1024
    )

    size_results.append({
        "Model": model_name,
        "FP32_File_KB": fp32_size,
        "INT8_File_KB": int8_size,
        "Reduction_Percent":
            (
                1 - int8_size / fp32_size
            ) * 100
    })


quantized_size_df = pd.DataFrame(
    size_results
)

quantized_size_df.round(2)

In [ ]:
# ==========================================
# 180. QUANTIZATION DATAFRAME'İNİ OLUŞTUR
# ==========================================

quantization_df = pd.DataFrame(
    quantization_results
)

quantization_df["RMSE_Change_Percent"] = (
    (
        quantization_df["INT8_RMSE"]
        - quantization_df["FP32_RMSE"]
    )
    /
    quantization_df["FP32_RMSE"]
    * 100
)

quantization_df["R2_Change"] = (
    quantization_df["INT8_R2"]
    - quantization_df["FP32_R2"]
)

quantization_df.round(4)

In [ ]:
# ==========================================
# 182. QUANTIZATION SONUÇ TABLOSU
# ==========================================

quantization_summary = quantization_df.merge(
    quantized_size_df,
    on="Model",
    how="inner"
)

quantization_summary["R2_Loss"] = (
    quantization_summary["FP32_R2"]
    - quantization_summary["INT8_R2"]
)

quantization_summary.round(4)

In [ ]:
quantization_summary.to_csv(
    "quantization_karsilastirma.csv",
    index=False
)

print("Kaydedildi: quantization_karsilastirma.csv")

In [ ]:
# ==========================================
# 183. FP32 - INT8 R² KARŞILAŞTIRMASI
# ==========================================

x = np.arange(len(quantization_summary))
width = 0.35

plt.figure(figsize=(9, 5))

plt.bar(
    x - width/2,
    quantization_summary["FP32_R2"],
    width,
    label="FP32"
)

plt.bar(
    x + width/2,
    quantization_summary["INT8_R2"],
    width,
    label="INT8"
)

plt.xticks(
    x,
    quantization_summary["Model"]
)

plt.xlabel("Model")
plt.ylabel("R² Skoru")
plt.title("FP32 ve INT8 Tahmin Performansının Karşılaştırılması")

plt.legend()
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()

plt.savefig(
    "FP32_INT8_R2_karsilastirma.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "FP32_INT8_R2_karsilastirma.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 184. FP32 - INT8 DOSYA BOYUTU
# ==========================================

x = np.arange(len(quantized_size_df))
width = 0.35

plt.figure(figsize=(9, 5))

plt.bar(
    x - width/2,
    quantized_size_df["FP32_File_KB"],
    width,
    label="FP32"
)

plt.bar(
    x + width/2,
    quantized_size_df["INT8_File_KB"],
    width,
    label="INT8"
)

plt.xticks(
    x,
    quantized_size_df["Model"]
)

plt.xlabel("Model")
plt.ylabel("Dosya Boyutu (KB)")
plt.title("FP32 ve INT8 Model Dosya Boyutlarının Karşılaştırılması")

plt.legend()
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()

plt.savefig(
    "FP32_INT8_boyut_karsilastirma.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "FP32_INT8_boyut_karsilastirma.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 185. ONNX KURULUM
# ==========================================

!pip install -q onnx onnxruntime
import onnx
import onnxruntime as ort

print("ONNX sürümü        :", onnx.__version__)
print("ONNX Runtime sürümü:", ort.__version__)

In [ ]:
# ==========================================
# 186. MLP -> ONNX
# ==========================================

mlp_deploy = deployment_models["MLP"]

mlp_deploy = mlp_deploy.cpu()
mlp_deploy.eval()

dummy_input = torch.randn(
    1,
    24,
    12,
    dtype=torch.float32
)

torch.onnx.export(
    mlp_deploy,
    dummy_input,
    "MLP_fp32.onnx",

    export_params=True,
    opset_version=17,
    do_constant_folding=True,

    input_names=["input"],
    output_names=["output"],

    dynamic_axes={
        "input": {
            0: "batch_size"
        },
        "output": {
            0: "batch_size"
        }
    },

    # Colab/PyTorch sürümleri arasında
    # daha kararlı legacy exporter
    dynamo=False
)

print("MLP_fp32.onnx başarıyla oluşturuldu.")

In [ ]:
# ==========================================
# 187. ONNX MODEL CHECK
# ==========================================

onnx_model = onnx.load(
    "MLP_fp32.onnx"
)

onnx.checker.check_model(
    onnx_model
)

print("ONNX model yapısal olarak geçerli.")

In [ ]:
# ==========================================
# 188. PYTORCH - ONNX SAYISAL DOĞRULAMA
# ==========================================

sample = X_test_tensor[0:1].cpu()

# -----------------------------
# PYTORCH
# -----------------------------

with torch.inference_mode():

    pytorch_output = (
        mlp_deploy(sample)
        .detach()
        .cpu()
        .numpy()
    )

# -----------------------------
# ONNX RUNTIME
# -----------------------------

session = ort.InferenceSession(
    "MLP_fp32.onnx",
    providers=[
        "CPUExecutionProvider"
    ]
)

onnx_output = session.run(
    ["output"],
    {
        "input":
        sample.numpy().astype(np.float32)
    }
)[0]

# -----------------------------
# KARŞILAŞTIR
# -----------------------------

difference = np.max(
    np.abs(
        pytorch_output -
        onnx_output
    )
)

print("PyTorch çıktı :", pytorch_output.flatten())
print("ONNX çıktı    :", onnx_output.flatten())

print(
    "\nMaksimum mutlak fark:",
    difference
)

In [ ]:
# ==========================================
# 189. ONNX - TÜM TEST SETİ
# ==========================================

onnx_predictions_scaled = session.run(
    ["output"],
    {
        "input":
        X_test.astype(np.float32)
    }
)[0]

# Gerçek ölçeğe dön
onnx_predictions = (
    target_scaler.inverse_transform(
        onnx_predictions_scaled
    )
    .flatten()
)

# Test gerçek değerleri
onnx_true = (
    target_scaler.inverse_transform(
        y_test_scaled
    )
    .flatten()
)

(
    onnx_mae,
    onnx_rmse,
    onnx_r2,
    onnx_mape,
    onnx_smape
) = calculate_metrics(
    onnx_true,
    onnx_predictions
)

print("=" * 60)
print("MLP ONNX TEST SONUÇLARI")
print("=" * 60)

print(f"MAE   : {onnx_mae:.6f}")
print(f"RMSE  : {onnx_rmse:.6f}")
print(f"R²    : {onnx_r2:.6f}")
print(f"MAPE  : {onnx_mape:.3f}%")
print(f"SMAPE : {onnx_smape:.3f}%")

In [ ]:
# ==========================================
# 190. TÜM TEST SETİNDE
# PYTORCH - ONNX ÇIKTI FARKI
# ==========================================

mlp_deploy.eval()

with torch.inference_mode():

    pytorch_all_scaled = (
        mlp_deploy(
            torch.tensor(
                X_test,
                dtype=torch.float32
            )
        )
        .cpu()
        .numpy()
    )

max_abs_difference = np.max(
    np.abs(
        pytorch_all_scaled
        - onnx_predictions_scaled
    )
)

mean_abs_difference = np.mean(
    np.abs(
        pytorch_all_scaled
        - onnx_predictions_scaled
    )
)

print(
    "Maksimum mutlak fark:",
    max_abs_difference
)

print(
    "Ortalama mutlak fark:",
    mean_abs_difference
)

In [ ]:
# ==========================================
# 191. ONNX DOSYA BOYUTU
# ==========================================

onnx_size_kb = (
    os.path.getsize(
        "MLP_fp32.onnx"
    )
    / 1024
)

print(
    f"MLP ONNX dosya boyutu: "
    f"{onnx_size_kb:.2f} KB"
)

In [ ]:
# ==========================================
# 192. FPGA İÇİN STATIC MLP ONNX
# ==========================================

mlp_deploy.eval()

dummy_input = torch.randn(
    1,
    24,
    12,
    dtype=torch.float32
)

torch.onnx.export(
    mlp_deploy,
    dummy_input,
    "MLP_fp32_static.onnx",

    export_params=True,
    opset_version=17,
    do_constant_folding=True,

    input_names=["input"],
    output_names=["output"],

    # FPGA için sabit giriş
    dynamic_axes=None,

    dynamo=False
)

print(
    "MLP_fp32_static.onnx oluşturuldu."
)

In [ ]:
# ==========================================
# 193. STATIC ONNX CHECK
# ==========================================

mlp_static_onnx = onnx.load(
    "MLP_fp32_static.onnx"
)

onnx.checker.check_model(
    mlp_static_onnx
)

print(
    "Static MLP ONNX yapısal olarak geçerli."
)

In [ ]:
# ==========================================
# 194. ONNX INPUT / OUTPUT SHAPE
# ==========================================

for input_tensor in mlp_static_onnx.graph.input:

    dims = []

    for dim in input_tensor.type.tensor_type.shape.dim:

        if dim.dim_value:
            dims.append(dim.dim_value)
        else:
            dims.append("?")

    print(
        "Input :",
        input_tensor.name,
        dims
    )


for output_tensor in mlp_static_onnx.graph.output:

    dims = []

    for dim in output_tensor.type.tensor_type.shape.dim:

        if dim.dim_value:
            dims.append(dim.dim_value)
        else:
            dims.append("?")

    print(
        "Output:",
        output_tensor.name,
        dims
    )

In [ ]:
# ==========================================
# 195. LSTM -> ONNX
# ==========================================

lstm_deploy = deployment_models["LSTM"]

lstm_deploy = lstm_deploy.cpu()
lstm_deploy.eval()

dummy_input = torch.randn(
    1,
    24,
    12,
    dtype=torch.float32
)

torch.onnx.export(
    lstm_deploy,
    dummy_input,
    "LSTM_fp32_static.onnx",

    export_params=True,
    opset_version=17,
    do_constant_folding=True,

    input_names=["input"],
    output_names=["output"],

    dynamic_axes=None,

    dynamo=False
)

print(
    "LSTM_fp32_static.onnx oluşturuldu."
)

In [ ]:
# ==========================================
# 196. LSTM ONNX CHECK
# ==========================================

lstm_onnx = onnx.load(
    "LSTM_fp32_static.onnx"
)

onnx.checker.check_model(
    lstm_onnx
)

print(
    "LSTM ONNX yapısal olarak geçerli."
)

In [ ]:
# ==========================================
# 197. LSTM PYTORCH - ONNX
# ==========================================

sample = X_test_tensor[0:1].cpu()

with torch.inference_mode():

    lstm_torch_output = (
        lstm_deploy(sample)
        .cpu()
        .numpy()
    )


lstm_session = ort.InferenceSession(
    "LSTM_fp32_static.onnx",
    providers=[
        "CPUExecutionProvider"
    ]
)

lstm_onnx_output = lstm_session.run(
    ["output"],
    {
        "input":
        sample.numpy().astype(np.float32)
    }
)[0]


lstm_difference = np.max(
    np.abs(
        lstm_torch_output
        - lstm_onnx_output
    )
)

print(
    "PyTorch:",
    lstm_torch_output.flatten()
)

print(
    "ONNX   :",
    lstm_onnx_output.flatten()
)

print(
    "Maksimum fark:",
    lstm_difference
)

In [ ]:
# ==========================================
# 198. GRU -> ONNX
# ==========================================

gru_deploy = deployment_models["GRU"]

gru_deploy = gru_deploy.cpu()
gru_deploy.eval()

torch.onnx.export(
    gru_deploy,
    dummy_input,
    "GRU_fp32_static.onnx",

    export_params=True,
    opset_version=17,
    do_constant_folding=True,

    input_names=["input"],
    output_names=["output"],

    dynamic_axes=None,

    dynamo=False
)

print(
    "GRU_fp32_static.onnx oluşturuldu."
)

In [ ]:
# ==========================================
# 199. GRU ONNX CHECK
# ==========================================

gru_onnx = onnx.load(
    "GRU_fp32_static.onnx"
)

onnx.checker.check_model(
    gru_onnx
)

print(
    "GRU ONNX yapısal olarak geçerli."
)

In [ ]:
# ==========================================
# 200. GRU PYTORCH - ONNX
# ==========================================

with torch.inference_mode():

    gru_torch_output = (
        gru_deploy(sample)
        .cpu()
        .numpy()
    )


gru_session = ort.InferenceSession(
    "GRU_fp32_static.onnx",
    providers=[
        "CPUExecutionProvider"
    ]
)

gru_onnx_output = gru_session.run(
    ["output"],
    {
        "input":
        sample.numpy().astype(np.float32)
    }
)[0]


gru_difference = np.max(
    np.abs(
        gru_torch_output
        - gru_onnx_output
    )
)

print(
    "PyTorch:",
    gru_torch_output.flatten()
)

print(
    "ONNX   :",
    gru_onnx_output.flatten()
)

print(
    "Maksimum fark:",
    gru_difference
)

In [ ]:
# ==========================================
# 201. ONNX OPERATOR ANALİZ FONKSİYONU
# ==========================================

from collections import Counter

def analyze_onnx_operations(
    onnx_path
):

    model = onnx.load(
        onnx_path
    )

    operations = [
        node.op_type
        for node in model.graph.node
    ]

    counts = Counter(
        operations
    )

    print("=" * 60)
    print(onnx_path)
    print("=" * 60)

    print(
        "Toplam ONNX node:",
        len(operations)
    )

    print("\nOperatörler:")

    for op, count in sorted(
        counts.items()
    ):

        print(
            f"{op:20s}: {count}"
        )

    return counts

In [ ]:
# ==========================================
# 202. MLP OPERATOR ANALİZİ
# ==========================================

mlp_ops = analyze_onnx_operations(
    "MLP_fp32_static.onnx"
)

In [ ]:
# ==========================================
# 203. LSTM OPERATOR ANALİZİ
# ==========================================

lstm_ops = analyze_onnx_operations(
    "LSTM_fp32_static.onnx"
)

In [ ]:
# ==========================================
# 204. GRU OPERATOR ANALİZİ
# ==========================================

gru_ops = analyze_onnx_operations(
    "GRU_fp32_static.onnx"
)

In [ ]:
# ==========================================
# 205. ONNX OPERATOR KARŞILAŞTIRMASI
# ==========================================

all_ops = sorted(
    set(mlp_ops.keys())
    | set(lstm_ops.keys())
    | set(gru_ops.keys())
)

operator_comparison = pd.DataFrame({
    "Operator": all_ops,

    "MLP": [
        mlp_ops.get(op, 0)
        for op in all_ops
    ],

    "LSTM": [
        lstm_ops.get(op, 0)
        for op in all_ops
    ],

    "GRU": [
        gru_ops.get(op, 0)
        for op in all_ops
    ]
})

operator_comparison

In [ ]:
operator_comparison.to_csv(
    "onnx_operator_karsilastirma.csv",
    index=False
)

In [ ]:
# ==========================================
# 206. HLS4ML KURULUMU
# ==========================================

!pip install -q hls4ml
import hls4ml

print("hls4ml sürümü:", hls4ml.__version__)


In [ ]:
# ==========================================
# 207. HLS TOOL KONTROLÜ
# ==========================================

import shutil

print(
    "vitis_hls:",
    shutil.which("vitis_hls")
)

print(
    "vivado_hls:",
    shutil.which("vivado_hls")
)

In [ ]:
# ==========================================
# 208. MLP HLS CONFIG
# ==========================================

mlp_hls_config = hls4ml.utils.config_from_pytorch_model(
    mlp_deploy,

    # Batch hariç giriş:
    # 24 saat x 12 özellik
    input_shape=(24, 12),

    granularity="name",

    backend="Vitis",

    default_precision="fixed<16,6>",

    default_reuse_factor=1
)

mlp_hls_config

In [ ]:
# ==========================================
# 209. HLS CONFIG İNCELEME
# ==========================================

import pprint

pprint.pp(
    mlp_hls_config
)

In [ ]:
# ==========================================
# 210. PYTORCH MLP -> HLS4ML
# ==========================================

mlp_hls_model = hls4ml.converters.convert_from_pytorch_model(
    mlp_deploy,

    hls_config=mlp_hls_config,

    output_dir="mlp_hls_project",

    project_name="mlp_energy",

    backend="Vitis",

    io_type="io_parallel",

    clock_period=5
)

print(
    "MLP hls4ml dönüşümü tamamlandı."
)

In [ ]:
# ==========================================
# 211. HLS MODEL ÖZET
# ==========================================

mlp_hls_model.config.config
mlp_hls_model.get_input_variables()
mlp_hls_model.get_output_variables()

In [ ]:
# ==========================================
# 212. HLS MODEL GRAFİĞİ
# ==========================================

hls4ml.utils.plot_model(
    mlp_hls_model,
    to_file="MLP_hls4ml_mimarisi.png",
    show_shapes=True,
    show_precision=True
)

In [ ]:
# ==========================================
# 213. HLS PROJECT WRITE
# ==========================================

mlp_hls_model.write()

print(
    "HLS proje dosyaları oluşturuldu."
)
import os

print(
    os.listdir(
        "/content/mlp_hls_project"
    )
)

In [ ]:
# ==========================================
# 214. GENERATED HLS FILES
# ==========================================

for root, dirs, files in os.walk(
    "/content/mlp_hls_project"
):

    level = (
        root.replace(
            "/content/mlp_hls_project",
            ""
        ).count(os.sep)
    )

    indent = "  " * level

    print(
        f"{indent}{os.path.basename(root)}/"
    )

    for file in files[:15]:

        print(
            f"{indent}  {file}"
        )

In [ ]:
mlp_hls_model.compile()
mlp_hls_model.build()

In [ ]:
# ==========================================
# 215. HLS FIXED<16,6> DOĞRULAMA
# ==========================================

# compile daha önce çalıştı ama tekrar çağrılması sorun değil
mlp_hls_model.compile()

# HLS giriş verisi
X_hls = np.ascontiguousarray(
    X_test.astype(np.float32)
)

# HLS tahmini
hls_pred_scaled = mlp_hls_model.predict(
    X_hls
)

hls_pred_scaled = np.asarray(
    hls_pred_scaled
).reshape(-1, 1)

print("HLS tahmin şekli:", hls_pred_scaled.shape)

In [ ]:
# ==========================================
# 216. PYTORCH FP32 vs HLS FIXED<16,6>
# ==========================================

mlp_deploy = mlp_deploy.cpu()
mlp_deploy.eval()

with torch.inference_mode():

    pytorch_pred_scaled = (
        mlp_deploy(
            torch.tensor(
                X_hls,
                dtype=torch.float32
            )
        )
        .cpu()
        .numpy()
        .reshape(-1, 1)
    )

difference = np.abs(
    pytorch_pred_scaled
    - hls_pred_scaled
)

print(
    "Maksimum ölçeklenmiş çıktı farkı:",
    difference.max()
)

print(
    "Ortalama ölçeklenmiş çıktı farkı:",
    difference.mean()
)

In [ ]:
# ==========================================
# 217. HLS FIXED<16,6> TEST SONUÇLARI
# ==========================================

hls_pred_real = (
    target_scaler.inverse_transform(
        hls_pred_scaled
    )
    .flatten()
)

hls_true_real = (
    target_scaler.inverse_transform(
        y_test_scaled
    )
    .flatten()
)

(
    hls_mae,
    hls_rmse,
    hls_r2,
    hls_mape,
    hls_smape
) = calculate_metrics(
    hls_true_real,
    hls_pred_real
)

print("=" * 60)
print("MLP HLS FIXED<16,6> TEST SONUÇLARI")
print("=" * 60)

print(f"MAE   : {hls_mae:.6f}")
print(f"RMSE  : {hls_rmse:.6f}")
print(f"R²    : {hls_r2:.6f}")
print(f"MAPE  : {hls_mape:.3f}%")
print(f"SMAPE : {hls_smape:.3f}%")

In [ ]:
# ==========================================
# 218. FIXED-POINT PRECISION SWEEP
# ==========================================

import shutil

PRECISION_CANDIDATES = [
    ("fixed<16,6>", 16, 6),
    ("fixed<12,5>", 12, 5),
    ("fixed<10,4>", 10, 4),
    ("fixed<8,4>",   8, 4)
]

precision_results = []

for precision, total_bits, integer_bits in PRECISION_CANDIDATES:

    print("\n" + "=" * 70)
    print("PRECISION:", precision)
    print("=" * 70)

    tag = f"w{total_bits}_i{integer_bits}"

    output_dir = f"/content/mlp_hls_{tag}"

    # Eski klasör varsa temizle
    shutil.rmtree(
        output_dir,
        ignore_errors=True
    )

    # ---------------------------------
    # CONFIG
    # ---------------------------------

    config = hls4ml.utils.config_from_pytorch_model(
        mlp_deploy,

        input_shape=(24, 12),

        granularity="name",

        backend="Vitis",

        default_precision=precision,

        default_reuse_factor=1
    )

    # ---------------------------------
    # HLS MODEL
    # ---------------------------------

    hls_model = hls4ml.converters.convert_from_pytorch_model(
        mlp_deploy,

        hls_config=config,

        output_dir=output_dir,

        project_name=f"mlp_{tag}",

        backend="Vitis",

        io_type="io_parallel",

        clock_period=5
    )

    # Vitis gerektirmeyen compile
    hls_model.compile()

    # ---------------------------------
    # TAHMİN
    # ---------------------------------

    pred_scaled = hls_model.predict(
        X_hls
    )

    pred_scaled = np.asarray(
        pred_scaled
    ).reshape(-1, 1)

    pred_real = (
        target_scaler.inverse_transform(
            pred_scaled
        )
        .flatten()
    )

    # ---------------------------------
    # METRİKLER
    # ---------------------------------

    (
        mae,
        rmse,
        r2,
        mape,
        smape
    ) = calculate_metrics(
        hls_true_real,
        pred_real
    )

    # FP32 çıkışına sayısal fark
    output_diff = np.abs(
        pytorch_pred_scaled
        - pred_scaled
    )

    precision_results.append({

        "Precision": precision,

        "Total_Bits": total_bits,

        "Integer_Bits": integer_bits,

        "Fractional_Bits":
            total_bits - integer_bits,

        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAPE": mape,
        "SMAPE": smape,

        "Max_Output_Diff":
            output_diff.max(),

        "Mean_Output_Diff":
            output_diff.mean()
    })

    print(f"RMSE  : {rmse:.6f}")
    print(f"R²    : {r2:.6f}")
    print(f"SMAPE : {smape:.3f}%")

In [ ]:
# ==========================================
# 219. PRECISION SWEEP TABLOSU
# ==========================================

precision_df = pd.DataFrame(
    precision_results
)

precision_df.round({
    "MAE": 4,
    "RMSE": 4,
    "R2": 4,
    "MAPE": 2,
    "SMAPE": 2,
    "Max_Output_Diff": 6,
    "Mean_Output_Diff": 6
})

In [ ]:
# ==========================================
# 220. FIXED-POINT PERFORMANS KAYBI
# ==========================================

FP32_REFERENCE_RMSE = 0.516787
FP32_REFERENCE_R2 = 0.459247

precision_df[
    "RMSE_Increase_Percent"
] = (
    (
        precision_df["RMSE"]
        - FP32_REFERENCE_RMSE
    )
    /
    FP32_REFERENCE_RMSE
    * 100
)

precision_df[
    "R2_Loss"
] = (
    FP32_REFERENCE_R2
    - precision_df["R2"]
)

precision_df.round(4)

In [ ]:
# ==========================================
# 221. KABUL EDİLEBİLİR PRECISION'LAR
# ==========================================

precision_df[
    "Acceptable"
] = (
    (
        precision_df[
            "RMSE_Increase_Percent"
        ] <= 1.0
    )
    &
    (
        precision_df[
            "R2_Loss"
        ] <= 0.01
    )
)

precision_df[
    [
        "Precision",
        "RMSE",
        "R2",
        "RMSE_Increase_Percent",
        "R2_Loss",
        "Acceptable"
    ]
].round(4)

In [ ]:
# ==========================================
# 222. PRECISION - RMSE
# ==========================================

plot_df = precision_df.sort_values(
    "Total_Bits"
)

plt.figure(figsize=(9, 5))

plt.plot(
    plot_df["Total_Bits"],
    plot_df["RMSE"],
    marker="o",
    linewidth=2
)

plt.axhline(
    FP32_REFERENCE_RMSE,
    linestyle="--",
    label="FP32 Referans"
)

plt.xlabel("Toplam Bit Genişliği")
plt.ylabel("RMSE (kW)")

plt.title(
    "Fixed-Point Bit Genişliğinin Tahmin Hatasına Etkisi"
)

plt.xticks(
    plot_df["Total_Bits"]
)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    "fixed_point_precision_RMSE.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "fixed_point_precision_RMSE.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 223. PRECISION - R²
# ==========================================

plt.figure(figsize=(9, 5))

plt.plot(
    plot_df["Total_Bits"],
    plot_df["R2"],
    marker="o",
    linewidth=2
)

plt.axhline(
    FP32_REFERENCE_R2,
    linestyle="--",
    label="FP32 Referans"
)

plt.xlabel("Toplam Bit Genişliği")
plt.ylabel("R²")

plt.title(
    "Fixed-Point Bit Genişliğinin R² Performansına Etkisi"
)

plt.xticks(
    plot_df["Total_Bits"]
)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    "fixed_point_precision_R2.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "fixed_point_precision_R2.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# 224. PRECISION SONUÇLARINI KAYDET
# ==========================================

precision_df.to_csv(
    "fixed_point_precision_sweep.csv",
    index=False
)

print(
    "Kaydedildi: fixed_point_precision_sweep.csv"
)

In [ ]:
# ==========================================
# 225. HLS PROJESİNİ ZIPLE
# ==========================================

import shutil
from google.colab import files

zip_path = shutil.make_archive(
    "/content/mlp_hls_project",
    "zip",
    "/content/mlp_hls_project"
)

print(
    "Hazır:",
    zip_path
)

files.download(
    zip_path
)

In [ ]:
# ==========================================
# 226. MLP AĞIRLIK / BIAS ARALIKLARI
# ==========================================

mlp_deploy.eval()

print("=" * 70)
print("MLP PARAMETRE ARALIKLARI")
print("=" * 70)

for name, param in mlp_deploy.named_parameters():

    data = param.detach().cpu().numpy()

    print(
        f"{name:30s} | "
        f"min={data.min(): .6f} | "
        f"max={data.max(): .6f} | "
        f"abs_max={np.abs(data).max(): .6f}"
    )

In [ ]:
# ==========================================
# 227. AKTİVASYON ARALIK ANALİZİ
# ==========================================

activation_ranges = {}

hooks = []

def make_hook(name):

    def hook(module, input, output):

        out = output.detach().cpu().numpy()

        if name not in activation_ranges:

            activation_ranges[name] = {
                "min": out.min(),
                "max": out.max()
            }

        else:

            activation_ranges[name]["min"] = min(
                activation_ranges[name]["min"],
                out.min()
            )

            activation_ranges[name]["max"] = max(
                activation_ranges[name]["max"],
                out.max()
            )

    return hook


for name, module in mlp_deploy.named_modules():

    if isinstance(
        module,
        (nn.Linear, nn.ReLU)
    ):

        hooks.append(
            module.register_forward_hook(
                make_hook(name)
            )
        )


# Test kümesini batch batch geçir
temp_loader = DataLoader(
    TensorDataset(
        X_test_tensor,
        y_test_tensor
    ),
    batch_size=256,
    shuffle=False
)

with torch.inference_mode():

    for X_batch, _ in temp_loader:

        _ = mlp_deploy(
            X_batch.cpu()
        )


for hook in hooks:
    hook.remove()


print("=" * 70)
print("AKTİVASYON ARALIKLARI")
print("=" * 70)

for name, values in activation_ranges.items():

    abs_max = max(
        abs(values["min"]),
        abs(values["max"])
    )

    print(
        f"{name:20s} | "
        f"min={values['min']: .6f} | "
        f"max={values['max']: .6f} | "
        f"abs_max={abs_max: .6f}"
    )

In [ ]:
# ==========================================
# 228. HIGH-PRECISION HLS SANITY CHECK
# ==========================================

high_precision = "fixed<32,12>"

config_hp = hls4ml.utils.config_from_pytorch_model(
    mlp_deploy,
    input_shape=(24, 12),
    granularity="name",
    backend="Vitis",
    default_precision=high_precision,
    default_reuse_factor=1
)

hls_hp = hls4ml.converters.convert_from_pytorch_model(
    mlp_deploy,
    hls_config=config_hp,
    output_dir="/content/mlp_hls_high_precision",
    project_name="mlp_high_precision",
    backend="Vitis",
    io_type="io_parallel",
    clock_period=5
)

hls_hp.compile()

hp_pred_scaled = np.asarray(
    hls_hp.predict(X_hls)
).reshape(-1, 1)

hp_pred_real = target_scaler.inverse_transform(
    hp_pred_scaled
).flatten()

(
    hp_mae,
    hp_rmse,
    hp_r2,
    hp_mape,
    hp_smape
) = calculate_metrics(
    hls_true_real,
    hp_pred_real
)

hp_diff = np.abs(
    pytorch_pred_scaled
    - hp_pred_scaled
)

print("=" * 60)
print("HLS FIXED<32,12> SANITY CHECK")
print("=" * 60)

print(f"RMSE : {hp_rmse:.6f}")
print(f"R²   : {hp_r2:.6f}")

print(
    "Max output farkı :",
    hp_diff.max()
)

print(
    "Mean output farkı:",
    hp_diff.mean()
)

In [ ]:
# ==========================================
# 229. MIXED-PRECISION SWEEP
# ==========================================

WEIGHT_PRECISIONS = [
    ("fixed<16,4>", 16),
    ("fixed<14,4>", 14),
    ("fixed<12,3>", 12),
    ("fixed<10,3>", 10),
    ("fixed<8,2>",   8)
]

mixed_results = []

for weight_precision, bits in WEIGHT_PRECISIONS:

    print("\n" + "=" * 70)
    print("WEIGHT PRECISION:", weight_precision)
    print("=" * 70)

    config = hls4ml.utils.config_from_pytorch_model(
        mlp_deploy,
        input_shape=(24, 12),
        granularity="name",
        backend="Vitis",
        default_precision="fixed<18,8>",
        default_reuse_factor=1
    )

    # Dense katmanlarını bul
    for layer_name, layer_cfg in config["LayerName"].items():

        if "Precision" not in layer_cfg:
            continue

        precision_cfg = layer_cfg["Precision"]

        # Dense katmanlarında bulunan alanları ayrı ayarla
        if "weight" in precision_cfg:

            precision_cfg["weight"] = (
                weight_precision
            )

        if "bias" in precision_cfg:

            precision_cfg["bias"] = (
                weight_precision
            )

        if "result" in precision_cfg:

            precision_cfg["result"] = (
                "fixed<18,8>"
            )

        if "accum" in precision_cfg:

            precision_cfg["accum"] = (
                "fixed<32,12>"
            )


    output_dir = (
        f"/content/mlp_mixed_{bits}bit"
    )

    shutil.rmtree(
        output_dir,
        ignore_errors=True
    )

    hls_model = hls4ml.converters.convert_from_pytorch_model(
        mlp_deploy,
        hls_config=config,
        output_dir=output_dir,
        project_name=f"mlp_mixed_{bits}",
        backend="Vitis",
        io_type="io_parallel",
        clock_period=5
    )

    hls_model.compile()

    pred_scaled = np.asarray(
        hls_model.predict(X_hls)
    ).reshape(-1, 1)

    pred_real = target_scaler.inverse_transform(
        pred_scaled
    ).flatten()

    (
        mae,
        rmse,
        r2,
        mape,
        smape
    ) = calculate_metrics(
        hls_true_real,
        pred_real
    )

    diff = np.abs(
        pytorch_pred_scaled
        - pred_scaled
    )

    mixed_results.append({
        "Weight_Precision":
            weight_precision,

        "Weight_Bits":
            bits,

        "Accumulator":
            "fixed<32,12>",

        "Result":
            "fixed<18,8>",

        "MAE":
            mae,

        "RMSE":
            rmse,

        "R2":
            r2,

        "MAPE":
            mape,

        "SMAPE":
            smape,

        "Max_Output_Diff":
            diff.max(),

        "Mean_Output_Diff":
            diff.mean()
    })

    print(
        f"RMSE : {rmse:.6f}"
    )

    print(
        f"R²   : {r2:.6f}"
    )

In [ ]:
# ==========================================
# 230. MIXED PRECISION SONUÇLARI
# ==========================================

mixed_precision_df = pd.DataFrame(
    mixed_results
)

mixed_precision_df[
    "RMSE_Increase_Percent"
] = (
    (
        mixed_precision_df["RMSE"]
        - FP32_REFERENCE_RMSE
    )
    /
    FP32_REFERENCE_RMSE
    * 100
)

mixed_precision_df[
    "R2_Loss"
] = (
    FP32_REFERENCE_R2
    - mixed_precision_df["R2"]
)

mixed_precision_df[
    "Acceptable"
] = (
    (
        mixed_precision_df[
            "RMSE_Increase_Percent"
        ] <= 1.0
    )
    &
    (
        mixed_precision_df[
            "R2_Loss"
        ] <= 0.01
    )
)

mixed_precision_df.round({
    "MAE": 4,
    "RMSE": 4,
    "R2": 4,
    "MAPE": 2,
    "SMAPE": 2,
    "Max_Output_Diff": 6,
    "Mean_Output_Diff": 6,
    "RMSE_Increase_Percent": 2,
    "R2_Loss": 4
})

In [ ]:
# ==========================================
# 231. HLS GİRİŞ ARALIĞI
# ==========================================

print("=" * 60)
print("HLS GİRDİ ARALIĞI")
print("=" * 60)

print("Minimum :", X_hls.min())
print("Maksimum:", X_hls.max())
print(
    "Abs max :",
    np.abs(X_hls).max()
)

In [ ]:
# ==========================================
# 233. RANGE-AWARE MIXED PRECISION SWEEP
# ==========================================

RANGE_AWARE_WEIGHTS = [
    ("fixed<16,2>", 16),
    ("fixed<14,2>", 14),
    ("fixed<12,2>", 12),
    ("fixed<10,2>", 10),
    ("fixed<8,2>",   8)
]

range_aware_results = []

for weight_precision, weight_bits in RANGE_AWARE_WEIGHTS:

    print("\n" + "=" * 70)
    print("WEIGHT PRECISION:", weight_precision)
    print("=" * 70)

    # ---------------------------------
    # CONFIG
    # ---------------------------------

    config = hls4ml.utils.config_from_pytorch_model(
        mlp_deploy,
        input_shape=(24, 12),
        granularity="name",
        backend="Vitis",

        # Genel güvenli başlangıç
        default_precision="fixed<18,4>",

        default_reuse_factor=1
    )

    # ---------------------------------
    # DENSE 1
    # network_1
    # ---------------------------------

    p = config["LayerName"]["network_1"]["Precision"]

    p["weight"] = weight_precision
    p["bias"]   = weight_precision

    p["accum"]  = "fixed<32,12>"
    p["result"] = "fixed<18,4>"

    # ---------------------------------
    # RELU 1
    # network_2
    # ---------------------------------

    config["LayerName"]["network_2"][
        "Precision"
    ]["result"] = "fixed<18,4>"

    # ---------------------------------
    # DENSE 2
    # network_3
    # ---------------------------------

    p = config["LayerName"]["network_3"]["Precision"]

    p["weight"] = weight_precision
    p["bias"]   = weight_precision

    p["accum"]  = "fixed<32,12>"
    p["result"] = "fixed<16,3>"

    # ---------------------------------
    # RELU 2
    # network_4
    # ---------------------------------

    config["LayerName"]["network_4"][
        "Precision"
    ]["result"] = "fixed<16,3>"

    # ---------------------------------
    # OUTPUT DENSE
    # network_5
    # ---------------------------------

    p = config["LayerName"]["network_5"]["Precision"]

    p["weight"] = weight_precision
    p["bias"]   = weight_precision

    p["accum"]  = "fixed<32,12>"
    p["result"] = "fixed<16,2>"

    # ---------------------------------
    # CONVERT
    # ---------------------------------

    output_dir = (
        f"/content/mlp_rangeaware_{weight_bits}bit"
    )

    shutil.rmtree(
        output_dir,
        ignore_errors=True
    )

    hls_model = hls4ml.converters.convert_from_pytorch_model(
        mlp_deploy,

        hls_config=config,

        output_dir=output_dir,

        project_name=(
            f"mlp_rangeaware_{weight_bits}"
        ),

        backend="Vitis",

        io_type="io_parallel",

        clock_period=5
    )

    hls_model.compile()

    # ---------------------------------
    # PREDICTION
    # ---------------------------------

    pred_scaled = np.asarray(
        hls_model.predict(X_hls)
    ).reshape(-1, 1)

    pred_real = (
        target_scaler
        .inverse_transform(pred_scaled)
        .flatten()
    )

    # ---------------------------------
    # METRICS
    # ---------------------------------

    (
        mae,
        rmse,
        r2,
        mape,
        smape
    ) = calculate_metrics(
        hls_true_real,
        pred_real
    )

    output_diff = np.abs(
        pytorch_pred_scaled
        - pred_scaled
    )

    rmse_increase = (
        (rmse - FP32_REFERENCE_RMSE)
        / FP32_REFERENCE_RMSE
        * 100
    )

    r2_loss = (
        FP32_REFERENCE_R2
        - r2
    )

    acceptable = (
        rmse_increase <= 1.0
        and
        r2_loss <= 0.01
    )

    range_aware_results.append({

        "Weight_Precision":
            weight_precision,

        "Weight_Bits":
            weight_bits,

        "Dense1_Result":
            "fixed<18,4>",

        "Dense2_Result":
            "fixed<16,3>",

        "Output_Result":
            "fixed<16,2>",

        "Accumulator":
            "fixed<32,12>",

        "MAE":
            mae,

        "RMSE":
            rmse,

        "R2":
            r2,

        "MAPE":
            mape,

        "SMAPE":
            smape,

        "RMSE_Increase_Percent":
            rmse_increase,

        "R2_Loss":
            r2_loss,

        "Max_Output_Diff":
            output_diff.max(),

        "Mean_Output_Diff":
            output_diff.mean(),

        "Acceptable":
            acceptable
    })

    print(f"RMSE        : {rmse:.6f}")
    print(f"R²          : {r2:.6f}")
    print(
        f"RMSE artışı : %{rmse_increase:.3f}"
    )
    print(
        f"R² kaybı    : {r2_loss:.6f}"
    )
    print(
        "Kabul       :",
        acceptable
    )

In [ ]:
# ==========================================
# 234. RANGE-AWARE SONUÇ TABLOSU
# ==========================================

range_aware_df = pd.DataFrame(
    range_aware_results
)

range_aware_df[
    [
        "Weight_Precision",
        "RMSE",
        "R2",
        "SMAPE",
        "RMSE_Increase_Percent",
        "R2_Loss",
        "Max_Output_Diff",
        "Acceptable"
    ]
].round(4)

In [ ]:
# ==========================================
# 235. EN DÜŞÜK KABUL EDİLEBİLİR BIT
# ==========================================

acceptable_df = range_aware_df[
    range_aware_df["Acceptable"]
].copy()

if len(acceptable_df) > 0:

    best_hw_precision = (
        acceptable_df.sort_values(
            "Weight_Bits"
        ).iloc[0]
    )

    print("=" * 60)
    print("FPGA PRECISION ADAYI")
    print("=" * 60)

    print(
        "Ağırlık precision:",
        best_hw_precision[
            "Weight_Precision"
        ]
    )

    print(
        f"RMSE: "
        f"{best_hw_precision['RMSE']:.6f}"
    )

    print(
        f"R²: "
        f"{best_hw_precision['R2']:.6f}"
    )

    print(
        f"RMSE artışı: "
        f"%{best_hw_precision['RMSE_Increase_Percent']:.3f}"
    )

else:

    print(
        "Henüz kabul kriterini sağlayan "
        "precision bulunamadı."
    )

In [ ]:
# ==========================================
# 236. FINAL WEIGHT PRECISION SWEEP
# ==========================================

FINAL_WEIGHT_PRECISIONS = [
    ("fixed<16,2>", 16, 2),
    ("fixed<15,2>", 15, 2),
    ("fixed<15,1>", 15, 1),
    ("fixed<14,1>", 14, 1),
    ("fixed<13,1>", 13, 1),
    ("fixed<12,1>", 12, 1)
]

final_precision_results = []

for weight_precision, weight_bits, integer_bits in FINAL_WEIGHT_PRECISIONS:

    print("\n" + "=" * 70)
    print("WEIGHT PRECISION:", weight_precision)
    print("=" * 70)

    config = hls4ml.utils.config_from_pytorch_model(
        mlp_deploy,
        input_shape=(24, 12),
        granularity="name",
        backend="Vitis",
        default_precision="fixed<18,4>",
        default_reuse_factor=1
    )

    # Dense 1
    p = config["LayerName"]["network_1"]["Precision"]
    p["weight"] = weight_precision
    p["bias"]   = weight_precision
    p["accum"]  = "fixed<32,12>"
    p["result"] = "fixed<18,4>"

    # ReLU 1
    config["LayerName"]["network_2"]["Precision"]["result"] = (
        "fixed<18,4>"
    )

    # Dense 2
    p = config["LayerName"]["network_3"]["Precision"]
    p["weight"] = weight_precision
    p["bias"]   = weight_precision
    p["accum"]  = "fixed<32,12>"
    p["result"] = "fixed<16,3>"

    # ReLU 2
    config["LayerName"]["network_4"]["Precision"]["result"] = (
        "fixed<16,3>"
    )

    # Output
    p = config["LayerName"]["network_5"]["Precision"]
    p["weight"] = weight_precision
    p["bias"]   = weight_precision
    p["accum"]  = "fixed<32,12>"
    p["result"] = "fixed<16,2>"

    output_dir = (
        f"/content/mlp_final_precision_"
        f"{weight_bits}_{integer_bits}"
    )

    shutil.rmtree(
        output_dir,
        ignore_errors=True
    )

    model_hls = hls4ml.converters.convert_from_pytorch_model(
        mlp_deploy,
        hls_config=config,
        output_dir=output_dir,
        project_name=f"mlp_p{weight_bits}_{integer_bits}",
        backend="Vitis",
        io_type="io_parallel",
        clock_period=5
    )

    model_hls.compile()

    pred_scaled = np.asarray(
        model_hls.predict(X_hls)
    ).reshape(-1, 1)

    pred_real = (
        target_scaler
        .inverse_transform(pred_scaled)
        .flatten()
    )

    (
        mae,
        rmse,
        r2,
        mape,
        smape
    ) = calculate_metrics(
        hls_true_real,
        pred_real
    )

    rmse_increase = (
        (rmse - FP32_REFERENCE_RMSE)
        / FP32_REFERENCE_RMSE
        * 100
    )

    r2_loss = (
        FP32_REFERENCE_R2 - r2
    )

    acceptable = (
        rmse_increase <= 1.0
        and r2_loss <= 0.01
    )

    final_precision_results.append({
        "Precision": weight_precision,
        "Total_Bits": weight_bits,
        "Integer_Bits": integer_bits,
        "Fractional_Bits": weight_bits - integer_bits,
        "RMSE": rmse,
        "R2": r2,
        "SMAPE": smape,
        "RMSE_Increase_Percent": rmse_increase,
        "R2_Loss": r2_loss,
        "Acceptable": acceptable
    })

    print(f"RMSE        : {rmse:.6f}")
    print(f"R²          : {r2:.6f}")
    print(f"RMSE artışı : %{rmse_increase:.3f}")
    print(f"R² kaybı    : {r2_loss:.6f}")
    print(f"Kabul       : {acceptable}")

In [ ]:
final_precision_df = pd.DataFrame(
    final_precision_results
)

final_precision_df.round(4)

In [ ]:
accepted = final_precision_df[
    final_precision_df["Acceptable"]
].copy()

if not accepted.empty:

    final_precision = (
        accepted
        .sort_values(
            ["Total_Bits", "Integer_Bits"]
        )
        .iloc[0]
    )

    print("=" * 60)
    print("NİHAİ FPGA AĞIRLIK PRECISION")
    print("=" * 60)

    print(
        "Precision:",
        final_precision["Precision"]
    )

    print(
        "RMSE:",
        round(final_precision["RMSE"], 6)
    )

    print(
        "R²:",
        round(final_precision["R2"], 6)
    )

else:
    print("16-bit çözüm kullanılmalı.")

In [ ]:
# ==========================================
# 237. FP32 VALIDATION REFERANSI
# ==========================================

X_hls_val = np.ascontiguousarray(
    X_val.astype(np.float32)
)

y_val_for_inverse = np.asarray(
    y_val
).reshape(-1, 1)

val_true_real = (
    target_scaler
    .inverse_transform(y_val_for_inverse)
    .flatten()
)

mlp_deploy.eval()

with torch.inference_mode():

    val_fp32_scaled = (
        mlp_deploy(
            torch.tensor(
                X_hls_val,
                dtype=torch.float32
            )
        )
        .cpu()
        .numpy()
        .reshape(-1, 1)
    )

val_fp32_real = (
    target_scaler
    .inverse_transform(val_fp32_scaled)
    .flatten()
)

(
    val_fp32_mae,
    val_fp32_rmse,
    val_fp32_r2,
    val_fp32_mape,
    val_fp32_smape
) = calculate_metrics(
    val_true_real,
    val_fp32_real
)

print("=" * 60)
print("FP32 MLP VALIDATION REFERANSI")
print("=" * 60)

print(f"MAE   : {val_fp32_mae:.6f}")
print(f"RMSE  : {val_fp32_rmse:.6f}")
print(f"R²    : {val_fp32_r2:.6f}")
print(f"MAPE  : {val_fp32_mape:.3f}%")
print(f"SMAPE : {val_fp32_smape:.3f}%")

In [ ]:
# ==========================================
# 238. VALIDATION PRECISION SWEEP
# ==========================================

VALIDATION_PRECISIONS = [
    ("fixed<16,2>", 16, 2),
    ("fixed<15,2>", 15, 2),
    ("fixed<15,1>", 15, 1),
    ("fixed<14,1>", 14, 1),
    ("fixed<13,1>", 13, 1),
    ("fixed<12,1>", 12, 1)
]

validation_precision_results = []

for weight_precision, total_bits, integer_bits in VALIDATION_PRECISIONS:

    print("\n" + "=" * 65)
    print("PRECISION:", weight_precision)
    print("=" * 65)

    config = hls4ml.utils.config_from_pytorch_model(
        mlp_deploy,
        input_shape=(24, 12),
        granularity="name",
        backend="Vitis",
        default_precision="fixed<18,4>",
        default_reuse_factor=1
    )

    # Dense 1
    p = config["LayerName"]["network_1"]["Precision"]
    p["weight"] = weight_precision
    p["bias"] = weight_precision
    p["accum"] = "fixed<32,12>"
    p["result"] = "fixed<18,4>"

    # ReLU 1
    config["LayerName"]["network_2"]["Precision"]["result"] = (
        "fixed<18,4>"
    )

    # Dense 2
    p = config["LayerName"]["network_3"]["Precision"]
    p["weight"] = weight_precision
    p["bias"] = weight_precision
    p["accum"] = "fixed<32,12>"
    p["result"] = "fixed<16,3>"

    # ReLU 2
    config["LayerName"]["network_4"]["Precision"]["result"] = (
        "fixed<16,3>"
    )

    # Output
    p = config["LayerName"]["network_5"]["Precision"]
    p["weight"] = weight_precision
    p["bias"] = weight_precision
    p["accum"] = "fixed<32,12>"
    p["result"] = "fixed<16,2>"

    output_dir = (
        f"/content/mlp_val_precision_"
        f"{total_bits}_{integer_bits}"
    )

    shutil.rmtree(
        output_dir,
        ignore_errors=True
    )

    hls_model_val = hls4ml.converters.convert_from_pytorch_model(
        mlp_deploy,
        hls_config=config,
        output_dir=output_dir,
        project_name=f"mlp_val_{total_bits}_{integer_bits}",
        backend="Vitis",
        io_type="io_parallel",
        clock_period=5
    )

    hls_model_val.compile()

    pred_scaled = np.asarray(
        hls_model_val.predict(X_hls_val)
    ).reshape(-1, 1)

    pred_real = (
        target_scaler
        .inverse_transform(pred_scaled)
        .flatten()
    )

    (
        mae,
        rmse,
        r2,
        mape,
        smape
    ) = calculate_metrics(
        val_true_real,
        pred_real
    )

    rmse_increase = (
        (rmse - val_fp32_rmse)
        / val_fp32_rmse
        * 100
    )

    r2_loss = (
        val_fp32_r2 - r2
    )

    acceptable = (
        rmse_increase <= 1.0
        and
        r2_loss <= 0.01
    )

    validation_precision_results.append({
        "Precision": weight_precision,
        "Total_Bits": total_bits,
        "Integer_Bits": integer_bits,
        "Fractional_Bits": total_bits - integer_bits,
        "RMSE": rmse,
        "R2": r2,
        "SMAPE": smape,
        "RMSE_Increase_Percent": rmse_increase,
        "R2_Loss": r2_loss,
        "Acceptable": acceptable
    })

    print(f"RMSE        : {rmse:.6f}")
    print(f"R²          : {r2:.6f}")
    print(f"RMSE artışı : %{rmse_increase:.3f}")
    print(f"R² kaybı    : {r2_loss:.6f}")
    print(f"Kabul       : {acceptable}")

In [ ]:
validation_precision_df = pd.DataFrame(
    validation_precision_results
)

display(
    validation_precision_df.round(4)
)

accepted_val = validation_precision_df[
    validation_precision_df["Acceptable"]
].copy()

if not accepted_val.empty:

    selected_precision = (
        accepted_val
        .sort_values(
            ["Total_Bits", "Integer_Bits"]
        )
        .iloc[0]
    )

    FINAL_HLS_PRECISION = (
        selected_precision["Precision"]
    )

    print("=" * 60)
    print("VALIDATION'A GÖRE NİHAİ FPGA PRECISION")
    print("=" * 60)

    print(
        "Precision:",
        FINAL_HLS_PRECISION
    )

    print(
        f"Validation RMSE: "
        f"{selected_precision['RMSE']:.6f}"
    )

    print(
        f"RMSE artışı: "
        f"%{selected_precision['RMSE_Increase_Percent']:.3f}"
    )

    print(
        f"R² kaybı: "
        f"{selected_precision['R2_Loss']:.6f}"
    )

else:

    print(
        "Kabul kriterini sağlayan "
        "precision bulunamadı."
    )

In [ ]:
# ==========================================
# 240. NİHAİ FPGA HLS PROJESİ
# ==========================================

import shutil
import hls4ml

FINAL_WEIGHT_PRECISION = "fixed<14,1>"

final_config = hls4ml.utils.config_from_pytorch_model(
    mlp_deploy,
    input_shape=(24, 12),
    granularity="name",
    backend="Vitis",
    default_precision="fixed<18,4>",
    default_reuse_factor=1
)

# -----------------------------
# Dense 1
# -----------------------------
p = final_config["LayerName"]["network_1"]["Precision"]

p["weight"] = FINAL_WEIGHT_PRECISION
p["bias"]   = FINAL_WEIGHT_PRECISION
p["accum"]  = "fixed<32,12>"
p["result"] = "fixed<18,4>"

# ReLU 1
final_config["LayerName"]["network_2"]["Precision"]["result"] = (
    "fixed<18,4>"
)

# -----------------------------
# Dense 2
# -----------------------------
p = final_config["LayerName"]["network_3"]["Precision"]

p["weight"] = FINAL_WEIGHT_PRECISION
p["bias"]   = FINAL_WEIGHT_PRECISION
p["accum"]  = "fixed<32,12>"
p["result"] = "fixed<16,3>"

# ReLU 2
final_config["LayerName"]["network_4"]["Precision"]["result"] = (
    "fixed<16,3>"
)

# -----------------------------
# Output
# -----------------------------
p = final_config["LayerName"]["network_5"]["Precision"]

p["weight"] = FINAL_WEIGHT_PRECISION
p["bias"]   = FINAL_WEIGHT_PRECISION
p["accum"]  = "fixed<32,12>"
p["result"] = "fixed<16,2>"

# -----------------------------
# PROJE
# -----------------------------

FINAL_HLS_DIR = "/content/mlp_fpga_final"

shutil.rmtree(
    FINAL_HLS_DIR,
    ignore_errors=True
)

final_hls_model = hls4ml.converters.convert_from_pytorch_model(
    mlp_deploy,
    hls_config=final_config,
    output_dir=FINAL_HLS_DIR,
    project_name="mlp_energy_final",
    backend="Vitis",
    io_type="io_parallel",
    clock_period=5
)

final_hls_model.compile()
final_hls_model.write()

print("Nihai HLS projesi oluşturuldu.")
print("Precision:", FINAL_WEIGHT_PRECISION)
print("Klasör:", FINAL_HLS_DIR)

In [ ]:
# ==========================================
# 241. NİHAİ HLS TEST DOĞRULAMASI
# ==========================================

final_hls_pred_scaled = np.asarray(
    final_hls_model.predict(X_hls)
).reshape(-1, 1)

final_hls_pred = (
    target_scaler
    .inverse_transform(final_hls_pred_scaled)
    .flatten()
)

(
    final_hls_mae,
    final_hls_rmse,
    final_hls_r2,
    final_hls_mape,
    final_hls_smape
) = calculate_metrics(
    hls_true_real,
    final_hls_pred
)

print("=" * 60)
print("NİHAİ FPGA-UYUMLU MLP SONUCU")
print("=" * 60)

print(f"Precision : {FINAL_WEIGHT_PRECISION}")
print(f"MAE       : {final_hls_mae:.6f}")
print(f"RMSE      : {final_hls_rmse:.6f}")
print(f"R²        : {final_hls_r2:.6f}")
print(f"MAPE      : {final_hls_mape:.3f}%")
print(f"SMAPE     : {final_hls_smape:.3f}%")

In [ ]:
# ==========================================
# 242. FPGA CONFIG KAYDI
# ==========================================

final_fpga_config = pd.DataFrame([{
    "Model": "MLP",
    "Sequence_Length": 24,
    "Input_Features": 12,

    "Weight_Precision": "fixed<14,1>",
    "Dense1_Result": "fixed<18,4>",
    "Dense2_Result": "fixed<16,3>",
    "Output_Result": "fixed<16,2>",
    "Accumulator": "fixed<32,12>",

    "ReuseFactor": 1,
    "Target_Clock_ns": 5,

    "FP32_Test_RMSE": 0.516787,
    "HLS_Test_RMSE": final_hls_rmse,

    "FP32_Test_R2": 0.459247,
    "HLS_Test_R2": final_hls_r2
}])

final_fpga_config.to_csv(
    "/content/final_fpga_config.csv",
    index=False
)

final_fpga_config

In [ ]:
# ==========================================
# 243. COLAB ÇIKTILARINI YEDEKLE
# ==========================================

import os
import glob
import shutil
from google.colab import files

backup_dir = "/content/2209A_colab_final"

shutil.rmtree(
    backup_dir,
    ignore_errors=True
)

os.makedirs(
    backup_dir,
    exist_ok=True
)

# Grafikler, tablolar, modeller, ONNX
patterns = [
    "*.png",
    "*.pdf",
    "*.csv",
    "*.onnx",
    "*.pth"
]

for pattern in patterns:

    for src in glob.glob(
        f"/content/{pattern}"
    ):

        shutil.copy2(
            src,
            backup_dir
        )

# Nihai HLS projesi
shutil.copytree(
    "/content/mlp_fpga_final",
    os.path.join(
        backup_dir,
        "mlp_fpga_final"
    )
)

zip_path = shutil.make_archive(
    "/content/2209A_colab_final",
    "zip",
    backup_dir
)

print("Yedek hazır:", zip_path)

files.download(zip_path)

In [ ]:
# Yalnızca kontrol yapar; veri değiştirmez ve model eğitmez.
for name in ["hourly", "train_df", "val_df", "test_df"]:
    obj = globals().get(name)

    if obj is None:
        print(f"{name}: Oturumda bulunamadı")
    else:
        print(
            f"{name}: {len(obj)} satır | "
            f"{obj.index.min()} → {obj.index.max()}"
        )

print("\nGeçerli dakika eşiği:",
      globals().get("MIN_VALID_MINUTES", "Tanımlı değil"))

for name in ["X_test", "y_test"]:
    obj = globals().get(name)
    print(f"{name} boyutu:",
          getattr(obj, "shape", "Oturumda bulunamadı"))